
# Stage 2 — Data Extraction and Standardization

## Stage 2E — Standardization Without Semantic Relabelling

**Project:** The Real Cost of Being an Ojol Driver: What’s Left After a Day on the Road?

This notebook converts source-defined quantitative observations into source-specific standardized files while preserving the original economic meaning, denominator, time basis, sampling scope, and evidence layer.

Stage 2E does **not**:
- construct project net operating earnings;
- inflation-adjust monetary values;
- model fuel costs;
- infer missing categories;
- treat retrospective recall as independent historical survey waves;
- convert driver-reported deductions into realized transaction deductions;
- pool incompatible sources into a single analytical dataset.

Cross-source comparability tagging is completed in Stage 2F.



## 1. Locate the repository and load locked Stage 0–1 controls

**Purpose:** locate the repository already available in the execution environment, load the locked measurement dictionary and source registry, and create only the standardized-data directory now required by Stage 2E.


In [1]:

from pathlib import Path
import csv
import os
from collections import Counter, defaultdict

REPO_NAME = "indonesia-ojol-driver-economics-analysis"

# Optional explicit override. In normal Colab execution this can be left unset.
explicit_repo = os.environ.get("OJOL_REPO_DIR", "").strip()

candidate_paths = []
if explicit_repo:
    candidate_paths.append(Path(explicit_repo))

candidate_paths.extend([
    Path("/content") / REPO_NAME,
    Path("/content/drive/MyDrive") / REPO_NAME,
    Path.cwd() / REPO_NAME,
    Path.cwd(),
])

def has_locked_controls(path):
    return (
        (path / "metadata" / "stage0_measurement_dictionary.csv").is_file()
        and (path / "metadata" / "stage1_source_registry.csv").is_file()
    )

detected_repo = next(
    (path.resolve() for path in candidate_paths if has_locked_controls(path)),
    None,
)

def read_csv(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

def write_csv(path, records, columns):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="raise")
        writer.writeheader()
        writer.writerows(records)

# Locked Stage 0 metric identifiers used only when the repository is not
# mounted in the analytical runtime. These are control identifiers, not
# replacements for the public metadata files.
LOCKED_STAGE0_PROJECT_METRICS = {
    "source_id",
    "observation_id",
    "value_provenance",
    "publisher",
    "retrieval_date",
    "observation_period_start",
    "observation_period_end",
    "geography",
    "platform",
    "service_type",
    "vehicle_type",
    "sample_size",
    "recruitment_method",
    "customer_facing_fare",
    "customer_service_fee",
    "customer_other_charges",
    "driver_gross_service_earnings",
    "driver_side_platform_deduction",
    "driver_other_deductions",
    "incentive_bonus",
    "tips",
    "driver_receipts_before_operating_cost",
    "fuel_cost",
    "fuel_volume",
    "fuel_price_per_unit",
    "fuel_efficiency",
    "maintenance_cost",
    "consumables_cost",
    "mobile_data_cost",
    "parking_tolls_cost",
    "other_operating_cost",
    "depreciation_cost",
    "financing_interest_fees",
    "financing_principal_cash",
    "loan_installment_undivided",
    "lease_rental_cost",
    "online_hours",
    "productive_or_engaged_hours",
    "working_hours_source_reported",
    "order_count",
    "paid_trip_distance_km",
    "pickup_distance_km",
    "deadheading_repositioning_km",
    "total_work_related_distance_km",
    "utilization_rate",
    "realized_driver_deduction_rate",
    "customer_to_driver_gap",
    "customer_to_driver_gap_rate",
    "net_operating_earnings_cash_basis",
    "net_operating_earnings_economic_basis",
    "net_earnings_per_online_hour",
    "net_earnings_per_productive_hour",
    "net_earnings_per_order",
    "net_earnings_per_km",
    "regulatory_deduction_rule",
    "official_implementation_statement",
    "platform_stated_deduction",
    "regulatory_reconciliation_status",
    "nominal_value_idr",
    "real_value_idr",
    "affordability_benchmark_value",
    "comparability_status",
}

LOCKED_STAGE1_SOURCE_IDS = {f"SRC{i:03d}" for i in range(1, 31)}

if detected_repo is not None:
    REPO_DIR = detected_repo
    METADATA_DIR = REPO_DIR / "metadata"
    STANDARDIZED_DIR = REPO_DIR / "data" / "standardized"

    measurement_dictionary = read_csv(
        METADATA_DIR / "stage0_measurement_dictionary.csv"
    )
    source_registry = read_csv(
        METADATA_DIR / "stage1_source_registry.csv"
    )

    project_metrics = {row["field_name"] for row in measurement_dictionary}
    registered_source_ids = {row["source_id"] for row in source_registry}
    CONTROL_MODE = "repository_controls"

else:
    # Analytical fallback only. It intentionally uses a separate workspace
    # instead of creating a fake repository directory.
    workspace_root = Path(
        os.environ.get("OJOL_STAGE2_WORKSPACE_ROOT", "/content")
    )
    REPO_DIR = workspace_root / "ojol_stage2_workspace"
    METADATA_DIR = REPO_DIR / "metadata"
    STANDARDIZED_DIR = REPO_DIR / "data" / "standardized"

    project_metrics = set(LOCKED_STAGE0_PROJECT_METRICS)
    registered_source_ids = set(LOCKED_STAGE1_SOURCE_IDS)
    measurement_dictionary = [
        {"field_name": name} for name in sorted(project_metrics)
    ]
    source_registry = [
        {"source_id": source_id} for source_id in sorted(registered_source_ids)
    ]
    CONTROL_MODE = "embedded_locked_controls"

METADATA_DIR.mkdir(parents=True, exist_ok=True)
STANDARDIZED_DIR.mkdir(parents=True, exist_ok=True)

# Sanity checks required by downstream Stage 2E cells.
required_metrics = {
    "driver_gross_service_earnings",
    "working_hours_source_reported",
    "order_count",
}
missing_required_metrics = sorted(required_metrics - project_metrics)
if missing_required_metrics:
    raise RuntimeError(
        "Locked Stage 0 controls are missing required metrics: "
        + ", ".join(missing_required_metrics)
    )

required_sources = {"SRC010", "SRC013", "SRC017", "SRC020", "SRC022",
                    "SRC025", "SRC026", "SRC027", "SRC028", "SRC029", "SRC030"}
missing_required_sources = sorted(required_sources - registered_source_ids)
if missing_required_sources:
    raise RuntimeError(
        "Locked Stage 1 controls are missing required source IDs: "
        + ", ".join(missing_required_sources)
    )

print(f"Control mode: {CONTROL_MODE}")
print(f"Working root: {REPO_DIR}")
print(f"Locked project metrics available: {len(project_metrics)}")
print(f"Registered Stage 1 source IDs available: {len(registered_source_ids)}")
print(f"Standardized output directory: {STANDARDIZED_DIR}")


Control mode: repository_controls
Working root: /content/indonesia-ojol-driver-economics-analysis
Locked project metrics available: 62
Registered Stage 1 source IDs available: 30
Standardized output directory: /content/indonesia-ojol-driver-economics-analysis/data/standardized



## 2. Define the Stage 2 standardized observation schema

**Purpose:** implement the Stage 2B architecture in a long-form observation schema. `source_metric_code` preserves a source-specific concept when the locked Stage 0 dictionary does not contain a safe project equivalent. `category_bound_unit` is an implementation field needed to interpret numerical bracket bounds without changing the original category text.


In [2]:

STANDARDIZED_COLUMNS = [
    "observation_id",
    "source_id",
    "source_locator",
    "source_metric_code",
    "source_metric_label",
    "source_value_text",
    "source_unit",
    "metric_family",
    "project_metric",
    "mapping_status",
    "value_numeric",
    "unit",
    "statistic_type",
    "category_label",
    "category_lower_bound",
    "category_upper_bound",
    "category_bound_unit",
    "metric_denominator_n",
    "denominator_definition",
    "sample_size",
    "sample_population",
    "observation_period_start",
    "observation_period_end",
    "temporal_evidence_status",
    "geography",
    "platform",
    "service_type",
    "vehicle_type",
    "observation_level",
    "evidence_type",
    "value_provenance",
    "earnings_layer",
    "value_time_basis",
    "working_time_basis",
    "distance_basis",
    "cost_boundary",
    "currency",
    "nominal_real_status",
    "comparability_use",
    "comparability_status",
    "transformation_applied",
    "transformation_formula",
    "extraction_notes",
]

schema_definitions = {
    "observation_id": ("identity", "string", "yes", "Unique standardized observation identifier.", "Must be unique across all Stage 2 standardized source files."),
    "source_id": ("identity", "string", "yes", "Canonical source identifier from the Stage 1 source registry.", "Must resolve to the Stage 1 registry."),
    "source_locator": ("traceability", "string", "yes", "Page, figure, table, or slide locator within the source.", "Must support reinspection of the source observation."),
    "source_metric_code": ("mapping", "string", "yes", "Stage 2 source-specific semantic code.", "May be narrower than a locked Stage 0 project metric."),
    "source_metric_label": ("original", "string", "yes", "Source-preserved metric/question label.", "Do not replace with a stronger project interpretation."),
    "source_value_text": ("original", "string", "yes", "Source-preserved value text.", "Retain wording/format sufficient to audit the normalized value."),
    "source_unit": ("original", "string", "yes", "Source-reported unit or basis.", "Preserve even when normalized unit is also recorded."),
    "metric_family": ("mapping", "string", "yes", "Broad project family such as earnings, deduction, activity, operating_cost, incentive, or context.", "Does not imply exact project-metric equivalence."),
    "project_metric": ("mapping", "string", "no", "Locked Stage 0 metric only when semantic mapping is defensible.", "Leave blank rather than force an ambiguous source metric."),
    "mapping_status": ("mapping", "string", "yes", "Status describing the source-to-project mapping.", "Used to distinguish exact, caveated, temporal-caveat, category, and unmapped records."),
    "value_numeric": ("value", "number", "yes", "Normalized numeric value of the reported observation.", "No imputation or inferred residual categories."),
    "unit": ("value", "string", "yes", "Normalized unit of value_numeric.", "Examples: percent, IDR/day, hours/day, orders/day."),
    "statistic_type": ("value", "string", "yes", "Statistic represented by the value.", "Examples: mean, share, source_reported_ratio, multi_response_share."),
    "category_label": ("distribution", "string", "no", "Source-preserved category or bracket label.", "Required for distribution/bracket observations."),
    "category_lower_bound": ("distribution", "number", "no", "Numeric lower bound when a category has one.", "Open/closed interval semantics remain in category_label."),
    "category_upper_bound": ("distribution", "number", "no", "Numeric upper bound when a category has one.", "Open/closed interval semantics remain in category_label."),
    "category_bound_unit": ("distribution", "string", "no", "Unit applying to category bounds.", "Implementation field added so monetary, time, order, distance, and percentage brackets are unambiguous."),
    "metric_denominator_n": ("denominator", "number", "no", "Observation-specific denominator where reported or defensibly identified.", "Distinct from total sample size."),
    "denominator_definition": ("denominator", "string", "yes", "Population/subgroup definition used by the metric.", "Unresolved denominators remain explicit."),
    "sample_size": ("sample", "number", "yes", "Overall relevant source sample size.", "Does not replace metric-specific denominator."),
    "sample_population": ("sample", "string", "yes", "Source sample/population description.", "Preserve sampling and inference limits."),
    "observation_period_start": ("time", "string", "no", "Beginning of the source-defined observation period.", "Partial dates may be retained when the source is not more precise."),
    "observation_period_end": ("time", "string", "no", "End of the source-defined observation period.", "Do not substitute publication or retrieval date."),
    "temporal_evidence_status": ("time", "string", "yes", "Temporal evidence class.", "Examples: contemporaneous, retrospective_recall, reference_period."),
    "geography": ("scope", "string", "yes", "Geographic coverage of the observation.", "Preserve source discrepancies rather than silently resolve them."),
    "platform": ("scope", "string", "yes", "Platform scope.", "Multi-platform values are not allocated to individual platforms."),
    "service_type": ("scope", "string", "yes", "Service scope.", "Mixed service observations are not silently split."),
    "vehicle_type": ("scope", "string", "yes", "Vehicle universe.", "Primary project universe is motorcycle."),
    "observation_level": ("scope", "string", "yes", "Sample summary, sample distribution, or subsample distribution.", "Supports denominator and inference interpretation."),
    "evidence_type": ("evidence", "string", "yes", "Evidence form such as survey_self_report, survey_perception, or survey_preference.", "Perceptions are not converted into realized economic values."),
    "value_provenance": ("evidence", "string", "yes", "Observed/source_reported/derived/modelled/scenario class.", "Stage 2E records here are source-reported values."),
    "earnings_layer": ("economics", "string", "no", "Gross, source-defined net, unspecified, or other earnings layer.", "Prevents generic income from being treated as project gross/net."),
    "value_time_basis": ("economics", "string", "no", "Day, week, month, or other period basis.", "Required for time-normalized interpretation."),
    "working_time_basis": ("economics", "string", "no", "Online, productive, source-reported unspecified, or other basis.", "Generic working time is never relabelled online/productive."),
    "distance_basis": ("economics", "string", "no", "Paid-trip, pickup, deadheading, total-work, or source-reported unspecified basis.", "Ambiguous distance is not mapped to total work-related distance."),
    "cost_boundary": ("economics", "string", "no", "Operating-cost boundary represented by the source value.", "Mixed fuel plus food/drink bundles remain outside project operating-cost totals."),
    "currency": ("currency", "string", "no", "Currency relevant to the observation or bracket.", "IDR for nominal Indonesian monetary observations."),
    "nominal_real_status": ("currency", "string", "no", "Nominal or real status.", "Stage 2 preserves nominal source values."),
    "comparability_use": ("comparability", "string", "no", "Intended analytical comparison.", "Completed in Stage 2F."),
    "comparability_status": ("comparability", "string", "no", "Comparison-specific comparability class.", "Intentionally left blank in Stage 2E."),
    "transformation_applied": ("transformation", "string", "yes", "Transformation performed in Stage 2E.", "Must be none for source-preserved Stage 2E records."),
    "transformation_formula": ("transformation", "string", "no", "Formula if a later documented transformation is performed.", "Blank in Stage 2E."),
    "extraction_notes": ("traceability", "string", "yes", "Analytical note preserving source-definition caveats.", "Must record material ambiguity without resolving it by assumption."),
}

schema_rows = []
for field in STANDARDIZED_COLUMNS:
    group, dtype, required, definition, rule = schema_definitions[field]
    schema_rows.append({
        "field_name": field,
        "field_group": group,
        "data_type": dtype,
        "required": required,
        "definition": definition,
        "stage2e_rule": rule,
    })

schema_path = METADATA_DIR / "stage2_extraction_schema.csv"
write_csv(
    schema_path,
    schema_rows,
    ["field_name", "field_group", "data_type", "required", "definition", "stage2e_rule"],
)

print(f"Wrote {schema_path.relative_to(REPO_DIR)} with {len(schema_rows)} fields.")


Wrote metadata/stage2_extraction_schema.csv with 43 fields.



## 3. Define source-to-project metric mapping rules

**Purpose:** document which source concepts can map to locked project metrics and which must remain source-defined. A blank `project_metric` is an intentional semantic safeguard, not missing analytical work.


In [3]:

mapping_rows = [
    {
        "source_metric_code": "explicit_gross_driver_earnings",
        "metric_family": "earnings",
        "project_metric": "driver_gross_service_earnings",
        "mapping_status": "mapped_exact",
        "standardization_rule": "Map only when the source explicitly identifies driver earnings as gross/before operating costs.",
        "blocked_interpretation": "Do not infer customer-facing fare, post-deduction receipt, or project net earnings.",
    },
    {
        "source_metric_code": "explicit_gross_driver_earnings_recalled",
        "metric_family": "earnings",
        "project_metric": "driver_gross_service_earnings",
        "mapping_status": "mapped_with_temporal_caveat",
        "standardization_rule": "Preserve gross semantics but classify historical values as retrospective recall.",
        "blocked_interpretation": "Do not treat recalled historical values as independent panel/survey waves.",
    },
    {
        "source_metric_code": "source_reported_daily_income_unspecified_layer",
        "metric_family": "earnings",
        "project_metric": "",
        "mapping_status": "source_defined_unmapped",
        "standardization_rule": "Preserve generic daily income when the source does not establish gross/net/receipt semantics.",
        "blocked_interpretation": "Do not relabel as driver gross service earnings, driver receipts, or project net earnings.",
    },
    {
        "source_metric_code": "source_defined_net_after_mixed_cost_bundle",
        "metric_family": "earnings",
        "project_metric": "",
        "mapping_status": "source_defined_unmapped",
        "standardization_rule": "Retain the source-defined net value and its source cost boundary.",
        "blocked_interpretation": "Do not classify as project net operating earnings when the source subtracts food/drink or omits required project cost components.",
    },
    {
        "source_metric_code": "source_defined_net_income_unspecified_cost_basis",
        "metric_family": "earnings",
        "project_metric": "",
        "mapping_status": "source_defined_unmapped",
        "standardization_rule": "Retain the source-defined net label when the exact deducted components are unresolved.",
        "blocked_interpretation": "Do not classify as project net operating earnings.",
    },
    {
        "source_metric_code": "fuel_food_daily_cost_bundle",
        "metric_family": "operating_cost",
        "project_metric": "",
        "mapping_status": "source_defined_unmapped",
        "standardization_rule": "Retain as a mixed source bundle with cost_boundary=mixed_work_personal.",
        "blocked_interpretation": "Do not include the full bundle in project operating costs because food/drink is personal consumption.",
    },
    {
        "source_metric_code": "fuel_food_cost_share_of_gross",
        "metric_family": "operating_cost",
        "project_metric": "",
        "mapping_status": "source_defined_unmapped",
        "standardization_rule": "Retain the source-reported ratio and mixed cost boundary.",
        "blocked_interpretation": "Do not report as project operating-cost share.",
    },
    {
        "source_metric_code": "driver_reported_deduction_rate_category",
        "metric_family": "deduction",
        "project_metric": "",
        "mapping_status": "source_defined_unmapped",
        "standardization_rule": "Retain as driver-reported deduction evidence and preserve the reported bracket/rate.",
        "blocked_interpretation": "Do not convert into realized_driver_deduction_rate without matched transaction evidence.",
    },
    {
        "source_metric_code": "completed_orders_per_day",
        "metric_family": "activity",
        "project_metric": "order_count",
        "mapping_status": "mapped_exact_or_category",
        "standardization_rule": "Map when the source explicitly describes completed orders.",
        "blocked_interpretation": "Do not assume cancellations or other excluded orders are included.",
    },
    {
        "source_metric_code": "daily_distance_source_reported",
        "metric_family": "activity",
        "project_metric": "",
        "mapping_status": "source_defined_unmapped",
        "standardization_rule": "Retain daily distance with distance_basis=source_reported_unspecified.",
        "blocked_interpretation": "Do not map to paid_trip_distance_km or total_work_related_distance_km unless the source defines the basis.",
    },
    {
        "source_metric_code": "generic_working_hours",
        "metric_family": "activity",
        "project_metric": "working_hours_source_reported",
        "mapping_status": "mapped_with_caveat",
        "standardization_rule": "Map generic working time only to the source-reported working-hours metric.",
        "blocked_interpretation": "Do not relabel as online_hours or productive_or_engaged_hours.",
    },
    {
        "source_metric_code": "workdays_per_week",
        "metric_family": "activity",
        "project_metric": "",
        "mapping_status": "source_defined_unmapped",
        "standardization_rule": "Retain source-specific workday frequency for later workload analysis.",
        "blocked_interpretation": "Do not invent a locked Stage 0 project metric solely to force mapping.",
    },
    {
        "source_metric_code": "bonus_program_reported_available",
        "metric_family": "incentive",
        "project_metric": "",
        "mapping_status": "source_defined_unmapped",
        "standardization_rule": "Retain prevalence of reported program availability.",
        "blocked_interpretation": "Do not map prevalence to the monetary incentive_bonus metric.",
    },
    {
        "source_metric_code": "bonus_receipt_frequency",
        "metric_family": "incentive",
        "project_metric": "",
        "mapping_status": "source_defined_unmapped",
        "standardization_rule": "Retain source-reported bonus receipt frequency and denominator.",
        "blocked_interpretation": "Do not infer incentive value or contribution to earnings.",
    },
    {
        "source_metric_code": "bonus_receipt_experience",
        "metric_family": "incentive",
        "project_metric": "",
        "mapping_status": "source_defined_unmapped",
        "standardization_rule": "Retain whether respondents report ever receiving an incentive/bonus.",
        "blocked_interpretation": "Do not map prevalence to a monetary incentive amount.",
    },
    {
        "source_metric_code": "promo_impact_perception",
        "metric_family": "context",
        "project_metric": "",
        "mapping_status": "source_defined_unmapped",
        "standardization_rule": "Retain as survey perception evidence.",
        "blocked_interpretation": "Do not interpret as an observed causal change in driver earnings, orders, or incentives.",
    },
    {
        "source_metric_code": "platform_loyalty_reason",
        "metric_family": "context",
        "project_metric": "",
        "mapping_status": "source_defined_unmapped",
        "standardization_rule": "Retain multi-response preference/reason shares.",
        "blocked_interpretation": "Do not treat reasons as realized economic effects.",
    },
    {
        "source_metric_code": "commission_platform_preference",
        "metric_family": "context",
        "project_metric": "",
        "mapping_status": "source_defined_unmapped",
        "standardization_rule": "Retain preference between stated commission scenarios.",
        "blocked_interpretation": "Do not treat scenario commission values as realized driver deductions.",
    },
    {
        "source_metric_code": "commission_service_difference_perception",
        "metric_family": "context",
        "project_metric": "",
        "mapping_status": "source_defined_unmapped",
        "standardization_rule": "Retain perceived differences in treatment/service.",
        "blocked_interpretation": "Do not infer realized causal effects of commission levels.",
    },
    {
        "source_metric_code": "perceived_difference_reason",
        "metric_family": "context",
        "project_metric": "",
        "mapping_status": "source_defined_unmapped",
        "standardization_rule": "Retain subgroup multi-response reasons.",
        "blocked_interpretation": "Do not generalize subgroup perception shares to the full driver population.",
    },
]

mapping_path = METADATA_DIR / "stage2_metric_mapping.csv"
write_csv(
    mapping_path,
    mapping_rows,
    [
        "source_metric_code",
        "metric_family",
        "project_metric",
        "mapping_status",
        "standardization_rule",
        "blocked_interpretation",
    ],
)

for row in mapping_rows:
    metric = row["project_metric"]
    if metric and metric not in project_metrics:
        raise RuntimeError(
            f"Stage 2 mapping references a project metric absent from the locked Stage 0 dictionary: {metric}"
        )

print(f"Wrote {mapping_path.relative_to(REPO_DIR)} with {len(mapping_rows)} mapping rules.")


Wrote metadata/stage2_metric_mapping.csv with 20 mapping rules.



## 4. Standardize SRC010 — Polling Institute 2022

**Purpose:** preserve the survey's daily-income distribution without forcing an unspecified income layer into gross, receipts, or net earnings. Historical pre-pandemic and pandemic responses remain retrospective recall.


In [4]:

def blank_row():
    return {column: "" for column in STANDARDIZED_COLUMNS}

def build_row(**kwargs):
    row = blank_row()
    row.update({
        "mapping_status": "source_defined_unmapped",
        "evidence_type": "survey_self_report",
        "value_provenance": "source_reported",
        "transformation_applied": "none",
    })
    row.update(kwargs)
    return row

SRC010_META = {
    "sample_size": 810,
    "sample_population": "App-based motorcycle driver/partner sample in 31 surveyed kabupaten/kota",
    "geography": "31 kabupaten/kota in Indonesia",
    "platform": "multi-platform",
    "service_type": "mixed",
    "vehicle_type": "motorcycle",
}

src010_rows = []
periods = [
    (
        "Before pandemic (before May 2020)",
        "retrospective_recall",
        [11.6, 54.4, 17.5, 2.6, 13.2, 0.6],
    ),
    (
        "During the pandemic",
        "retrospective_recall",
        [66.0, 26.2, 1.9, 0.4, 3.1, 2.5],
    ),
    (
        "During 2022 up to survey fieldwork",
        "contemporaneous",
        [40.4, 52.5, 5.4, 1.2, 0.0, 0.5],
    ),
]

categories = [
    ("< Rp100,000", "", 100000, "IDR/day"),
    ("> Rp100,000 - Rp250,000", 100000, 250000, "IDR/day"),
    ("> Rp250,000 - Rp500,000", 250000, 500000, "IDR/day"),
    ("> Rp500,000", 500000, "", "IDR/day"),
    ("Not yet a driver/partner", "", "", ""),
    ("TT/TJ", "", "", ""),
]

observation_number = 1
for period_label, temporal_status, values in periods:
    for (category, lower, upper, bound_unit), value in zip(categories, values):
        src010_rows.append(
            build_row(
                observation_id=f"SRC010-O{observation_number:03d}",
                source_id="SRC010",
                source_locator='Slide "Penghasilan Menjadi Mitra Dalam Satu Hari"',
                source_metric_code="source_reported_daily_income_unspecified_layer",
                source_metric_label=(
                    "Berapa banyak penghasilan yang didapatkan dengan menjadi "
                    "mitra pengemudi/pengiriman dalam satu hari?"
                ),
                source_value_text=f"{value}%",
                source_unit="percent",
                metric_family="earnings",
                project_metric="",
                mapping_status="source_defined_unmapped",
                value_numeric=value,
                unit="percent",
                statistic_type="share",
                category_label=category,
                category_lower_bound=lower,
                category_upper_bound=upper,
                category_bound_unit=bound_unit,
                metric_denominator_n=810,
                denominator_definition=(
                    "Full driver sample; source retains not-yet-driver and "
                    "nonresponse categories for recalled periods"
                ),
                observation_period_start="",
                observation_period_end="",
                temporal_evidence_status=temporal_status,
                observation_level="sample_distribution",
                earnings_layer="unspecified",
                value_time_basis="day",
                currency="IDR",
                nominal_real_status="nominal",
                extraction_notes=(
                    f"Source-defined period: {period_label}. Income layer is not "
                    "identified as gross, receipt, or net."
                ),
                **SRC010_META,
            )
        )
        observation_number += 1

src010_path = STANDARDIZED_DIR / "src010_polling_institute_2022_driver_observations.csv"
write_csv(src010_path, src010_rows, STANDARDIZED_COLUMNS)

print(f"Wrote {src010_path.relative_to(REPO_DIR)} with {len(src010_rows)} observations.")


Wrote data/standardized/src010_polling_institute_2022_driver_observations.csv with 18 observations.



## 5. Standardize SRC013 — IDEAS 2025 driver survey

**Purpose:** standardize the 2025 earnings, mixed cost bundle, driver-reported deductions, bonus prevalence, orders, distance, working time, and workdays while retaining the source's 62-versus-67 locality discrepancy and all semantic boundaries.


In [5]:

SRC013_META = {
    "sample_size": 1018,
    "sample_population": "Ojol drivers in the 2025 IDEAS survey; non-probability purposive sample",
    "geography": "Indonesia; source reports 62 and 67 kabupaten/kota in different sections",
    "platform": "multi-platform",
    "service_type": "mixed",
    "vehicle_type": "motorcycle",
}

src013_rows = []

def add_src013(**kwargs):
    kwargs.setdefault("source_id", "SRC013")
    kwargs.setdefault("observation_id", f"SRC013-O{len(src013_rows)+1:03d}")
    for key, value in SRC013_META.items():
        kwargs.setdefault(key, value)
    src013_rows.append(build_row(**kwargs))

add_src013(
    source_locator="Policy brief pp. 5-6",
    source_metric_code="explicit_gross_driver_earnings",
    source_metric_label="Rerata pendapatan kotor harian ojol",
    source_value_text="Rp126 ribu per hari",
    source_unit="IDR/day",
    metric_family="earnings",
    project_metric="driver_gross_service_earnings",
    mapping_status="mapped_exact",
    value_numeric=126000,
    unit="IDR/day",
    statistic_type="mean",
    metric_denominator_n="",
    denominator_definition="2025 survey respondent base; metric-specific n not stated in narrative",
    observation_period_start="2025-12-01",
    observation_period_end="2025-12-15",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_summary",
    earnings_layer="gross_service_earnings",
    value_time_basis="day",
    currency="IDR",
    nominal_real_status="nominal",
    extraction_notes="Explicitly labelled gross daily earnings by the source.",
)

add_src013(
    source_locator="Policy brief pp. 5-6",
    source_metric_code="fuel_food_daily_cost_bundle",
    source_metric_label="Rerata biaya operasional harian: bensin dan makan-minum",
    source_value_text="Rp58 ribu per hari",
    source_unit="IDR/day",
    metric_family="operating_cost",
    value_numeric=58000,
    unit="IDR/day",
    statistic_type="mean",
    metric_denominator_n="",
    denominator_definition="2025 survey respondent base; metric-specific n not stated in narrative",
    observation_period_start="2025-12-01",
    observation_period_end="2025-12-15",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_summary",
    value_time_basis="day",
    cost_boundary="mixed_work_personal",
    currency="IDR",
    nominal_real_status="nominal",
    extraction_notes="Source bundle includes fuel and food/drink; it is not project operating cost.",
)

add_src013(
    source_locator="Policy brief pp. 5-6",
    source_metric_code="fuel_food_cost_share_of_gross",
    source_metric_label="Biaya operasional harian sebagai persentase pendapatan kotor harian",
    source_value_text="46%",
    source_unit="percent",
    metric_family="operating_cost",
    value_numeric=46.0,
    unit="percent",
    statistic_type="source_reported_ratio",
    metric_denominator_n="",
    denominator_definition="Source-reported ratio for the 2025 survey",
    observation_period_start="2025-12-01",
    observation_period_end="2025-12-15",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_summary",
    cost_boundary="mixed_work_personal",
    extraction_notes=(
        "Ratio uses the source's fuel-plus-food/drink bundle and must not be "
        "treated as project operating-cost share."
    ),
)

add_src013(
    source_locator="Policy brief pp. 5-6",
    source_metric_code="source_defined_net_after_mixed_cost_bundle",
    source_metric_label="Pendapatan bersih ojol per bulan",
    source_value_text="Rp1,7 juta per bulan",
    source_unit="IDR/month",
    metric_family="earnings",
    value_numeric=1700000,
    unit="IDR/month",
    statistic_type="source_reported_estimate",
    metric_denominator_n="",
    denominator_definition="Source-defined monthly estimate; metric-specific n not stated",
    observation_period_start="2025-12-01",
    observation_period_end="2025-12-15",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_summary",
    earnings_layer="source_defined_net",
    value_time_basis="month",
    cost_boundary="mixed_work_personal",
    currency="IDR",
    nominal_real_status="nominal",
    extraction_notes=(
        "Source-defined net uses a cost boundary that includes food/drink and "
        "is not project net operating earnings."
    ),
)

for category_label, lower, upper, share in [
    ("20% deduction", 20, 20, 50.3),
    ("25-30% deduction", 25, 30, 24.2),
]:
    add_src013(
        source_locator="Policy brief p. 6",
        source_metric_code="driver_reported_deduction_rate_category",
        source_metric_label=f"Ojol reporting application deduction of {category_label}",
        source_value_text=f"{share}%",
        source_unit="percent",
        metric_family="deduction",
        value_numeric=share,
        unit="percent",
        statistic_type="share",
        category_label=category_label,
        category_lower_bound=lower,
        category_upper_bound=upper,
        category_bound_unit="percent",
        metric_denominator_n=1018,
        denominator_definition="Full 2025 survey sample",
        observation_period_start="2025-12-01",
        observation_period_end="2025-12-15",
        temporal_evidence_status="contemporaneous",
        observation_level="sample_distribution",
        extraction_notes="Driver-reported deduction category; not transaction-level realized deduction.",
    )

add_src013(
    source_locator="Policy brief p. 8",
    source_metric_code="bonus_program_reported_available",
    source_metric_label="Aplikator masih memberi bonus (reward/poin)",
    source_value_text="62.6%",
    source_unit="percent",
    metric_family="incentive",
    value_numeric=62.6,
    unit="percent",
    statistic_type="share",
    metric_denominator_n=1018,
    denominator_definition=(
        "Full 2025 survey sample; 637 respondents reported that a bonus program remained available"
    ),
    observation_period_start="2025-12-01",
    observation_period_end="2025-12-15",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_distribution",
    extraction_notes="Program availability prevalence; not a monetary incentive amount.",
)

for category_label, share in [
    ("Never receives bonus", 30.9),
    ("Very rarely receives bonus", 42.7),
]:
    add_src013(
        source_locator="Policy brief p. 8",
        source_metric_code="bonus_receipt_frequency",
        source_metric_label="Frequency of receiving bonus among respondents reporting a bonus program",
        source_value_text=f"{share}%",
        source_unit="percent",
        metric_family="incentive",
        value_numeric=share,
        unit="percent",
        statistic_type="share",
        category_label=category_label,
        metric_denominator_n=637,
        denominator_definition="Respondents who reported that the platform still offered a bonus program",
        observation_period_start="2025-12-01",
        observation_period_end="2025-12-15",
        temporal_evidence_status="contemporaneous",
        observation_level="subsample_distribution",
        extraction_notes="Frequency prevalence within the bonus-program subgroup; not a monetary incentive amount.",
    )

add_src013(
    source_locator="Policy brief p. 9",
    source_metric_code="bonus_receipt_frequency",
    source_metric_label="Respondents able to receive bonus every day",
    source_value_text="2.4%",
    source_unit="percent",
    metric_family="incentive",
    value_numeric=2.4,
    unit="percent",
    statistic_type="share",
    category_label="Receives bonus every day",
    metric_denominator_n="",
    denominator_definition=(
        "Source wording says respondents; relationship to the 637-person "
        "bonus-program subgroup is not explicit in the narrative"
    ),
    observation_period_start="2025-12-01",
    observation_period_end="2025-12-15",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_distribution",
    extraction_notes="Denominator intentionally remains unresolved; not a monetary incentive amount.",
)

for category_label, lower, upper, share in [
    ("6-10 orders/day", 6, 10, 52.8),
    ("11-15 orders/day", 11, 15, 21.7),
    (">15 orders/day", 15, "", 10.3),
]:
    add_src013(
        source_locator="Policy brief p. 9",
        source_metric_code="completed_orders_per_day",
        source_metric_label="Jumlah order yang diselesaikan per hari",
        source_value_text=f"{share}%",
        source_unit="percent",
        metric_family="activity",
        project_metric="order_count",
        mapping_status="mapped_with_source_category",
        value_numeric=share,
        unit="percent",
        statistic_type="share",
        category_label=category_label,
        category_lower_bound=lower,
        category_upper_bound=upper,
        category_bound_unit="orders/day",
        metric_denominator_n=1018,
        denominator_definition="Full 2025 survey sample",
        observation_period_start="2025-12-01",
        observation_period_end="2025-12-15",
        temporal_evidence_status="contemporaneous",
        observation_level="sample_distribution",
        value_time_basis="day",
        extraction_notes="Source explicitly refers to completed orders. Unreported categories are not inferred.",
    )

for category_label, lower, upper, share in [
    ("21-40 km/day", 21, 40, 15.8),
    ("41-60 km/day", 41, 60, 43.6),
    ("61-100 km/day", 61, 100, 27.3),
]:
    add_src013(
        source_locator="Policy brief p. 9",
        source_metric_code="daily_distance_source_reported",
        source_metric_label="Jarak tempuh per hari",
        source_value_text=f"{share}%",
        source_unit="percent",
        metric_family="activity",
        value_numeric=share,
        unit="percent",
        statistic_type="share",
        category_label=category_label,
        category_lower_bound=lower,
        category_upper_bound=upper,
        category_bound_unit="km/day",
        metric_denominator_n=1018,
        denominator_definition="Full 2025 survey sample",
        observation_period_start="2025-12-01",
        observation_period_end="2025-12-15",
        temporal_evidence_status="contemporaneous",
        observation_level="sample_distribution",
        value_time_basis="day",
        distance_basis="source_reported_unspecified",
        extraction_notes=(
            "Source does not establish whether distance is paid-trip or total "
            "work-related distance; project metric is left unmapped."
        ),
    )

add_src013(
    source_locator="Policy brief pp. 9-10",
    source_metric_code="generic_working_hours",
    source_metric_label="Waktu kerja 9-12 jam per hari",
    source_value_text="51.0%",
    source_unit="percent",
    metric_family="activity",
    project_metric="working_hours_source_reported",
    mapping_status="mapped_with_caveat",
    value_numeric=51.0,
    unit="percent",
    statistic_type="share",
    category_label="9-12 hours/day",
    category_lower_bound=9,
    category_upper_bound=12,
    category_bound_unit="hours/day",
    metric_denominator_n=1018,
    denominator_definition="Full 2025 survey sample",
    observation_period_start="2025-12-01",
    observation_period_end="2025-12-15",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_distribution",
    value_time_basis="day",
    working_time_basis="source_reported_unspecified",
    extraction_notes="Generic source working time is not relabelled as online or productive time.",
)

add_src013(
    source_locator="Policy brief pp. 9-10",
    source_metric_code="workdays_per_week",
    source_metric_label="7 hari kerja tanpa libur",
    source_value_text="55.5%",
    source_unit="percent",
    metric_family="activity",
    value_numeric=55.5,
    unit="percent",
    statistic_type="share",
    category_label="7 workdays/week",
    category_lower_bound=7,
    category_upper_bound=7,
    category_bound_unit="days/week",
    metric_denominator_n=1018,
    denominator_definition="Full 2025 survey sample",
    observation_period_start="2025-12-01",
    observation_period_end="2025-12-15",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_distribution",
    value_time_basis="week",
    extraction_notes="Source-specific workday metric retained without creating a new locked project metric.",
)

src013_path = STANDARDIZED_DIR / "src013_ideas_2025_driver_observations.csv"
write_csv(src013_path, src013_rows, STANDARDIZED_COLUMNS)

print(f"Wrote {src013_path.relative_to(REPO_DIR)} with {len(src013_rows)} observations.")


Wrote data/standardized/src013_ideas_2025_driver_observations.csv with 18 observations.



## 6. Standardize SRC029 — IDEAS 2023 driver survey

**Purpose:** preserve current 2022–2023 metrics separately from longer-tenure retrospective recall, retain the mixed fuel-plus-food cost boundary, and keep working-time and distance semantics source-defined where necessary.


In [6]:

SRC029_META = {
    "sample_size": 225,
    "sample_population": "Ojol drivers at 10 transport nodes in Jabodetabek; non-probability purposive sample",
    "geography": "Jabodetabek",
    "platform": "multi-platform",
    "service_type": "mixed",
    "vehicle_type": "motorcycle",
}

src029_rows = []

def add_src029(**kwargs):
    kwargs.setdefault("source_id", "SRC029")
    kwargs.setdefault("observation_id", f"SRC029-O{len(src029_rows)+1:03d}")
    for key, value in SRC029_META.items():
        kwargs.setdefault(key, value)
    src029_rows.append(build_row(**kwargs))

add_src029(
    source_locator="Policy brief p. 3",
    source_metric_code="explicit_gross_driver_earnings",
    source_metric_label="Rerata pendapatan kotor harian ojek daring pada 2022-2023",
    source_value_text="Rp168 ribu per hari",
    source_unit="IDR/day",
    metric_family="earnings",
    project_metric="driver_gross_service_earnings",
    mapping_status="mapped_exact",
    value_numeric=168000,
    unit="IDR/day",
    statistic_type="mean",
    metric_denominator_n=186,
    denominator_definition="Respondents included in the main fair-pay summary table",
    observation_period_start="2022",
    observation_period_end="2023",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_summary",
    earnings_layer="gross_service_earnings",
    value_time_basis="day",
    currency="IDR",
    nominal_real_status="nominal",
    extraction_notes="Current-period gross daily earnings; distinct from longer-tenure recall series.",
)

for start_period, end_period, label, value in [
    ("2018", "2019", "Pre-pandemic recalled gross daily earnings", 305000),
    ("2020", "2021", "Pandemic recalled gross daily earnings", 100000),
    ("2022", "2023", "Post-pandemic recalled gross daily earnings among longer-tenure drivers", 175000),
]:
    add_src029(
        source_locator="Policy brief p. 3",
        source_metric_code="explicit_gross_driver_earnings_recalled",
        source_metric_label=label,
        source_value_text=f"Rp{value:,} per day".replace(",", "."),
        source_unit="IDR/day",
        metric_family="earnings",
        project_metric="driver_gross_service_earnings",
        mapping_status="mapped_with_temporal_caveat",
        value_numeric=value,
        unit="IDR/day",
        statistic_type="mean",
        metric_denominator_n="",
        denominator_definition="Drivers with more than four years of tenure; exact subgroup n not stated in extracted text",
        observation_period_start=start_period,
        observation_period_end=end_period,
        temporal_evidence_status="retrospective_recall",
        observation_level="subsample_summary",
        earnings_layer="gross_service_earnings",
        value_time_basis="day",
        currency="IDR",
        nominal_real_status="nominal",
        extraction_notes="Retrospective recall collected in the 2023 survey; not an independent historical survey wave.",
    )

add_src029(
    source_locator="Policy brief p. 3",
    source_metric_code="fuel_food_daily_cost_bundle",
    source_metric_label="Rerata beban operasional harian: bahan bakar dan makan-minum",
    source_value_text="Rp53 ribu per hari",
    source_unit="IDR/day",
    metric_family="operating_cost",
    value_numeric=53000,
    unit="IDR/day",
    statistic_type="mean",
    metric_denominator_n=186,
    denominator_definition="Respondents included in the main fair-pay summary table",
    observation_period_start="2022",
    observation_period_end="2023",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_summary",
    value_time_basis="day",
    cost_boundary="mixed_work_personal",
    currency="IDR",
    nominal_real_status="nominal",
    extraction_notes="Source bundle includes fuel and food/drink; it is not project operating cost.",
)

add_src029(
    source_locator="Policy brief p. 3",
    source_metric_code="fuel_food_cost_share_of_gross",
    source_metric_label="Beban operasional harian sebagai persentase pendapatan kotor harian",
    source_value_text="31%",
    source_unit="percent",
    metric_family="operating_cost",
    value_numeric=31.0,
    unit="percent",
    statistic_type="source_reported_ratio",
    metric_denominator_n=186,
    denominator_definition="Respondents included in the main fair-pay summary table",
    observation_period_start="2022",
    observation_period_end="2023",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_summary",
    cost_boundary="mixed_work_personal",
    extraction_notes="Source ratio uses fuel plus food/drink; not a project operating-cost ratio.",
)

add_src029(
    source_locator="Policy brief pp. 3-4",
    source_metric_code="completed_orders_per_day",
    source_metric_label="Rerata order diselesaikan per hari",
    source_value_text="10 orders/day",
    source_unit="orders/day",
    metric_family="activity",
    project_metric="order_count",
    mapping_status="mapped_exact",
    value_numeric=10,
    unit="orders/day",
    statistic_type="mean",
    metric_denominator_n=186,
    denominator_definition="Respondents included in the main fair-pay summary table",
    observation_period_start="2022",
    observation_period_end="2023",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_summary",
    value_time_basis="day",
    extraction_notes="Source explicitly describes completed orders.",
)

add_src029(
    source_locator="Policy brief pp. 3-4",
    source_metric_code="daily_distance_source_reported",
    source_metric_label="Rerata jarak tempuh per hari",
    source_value_text="42 km/day",
    source_unit="km/day",
    metric_family="activity",
    value_numeric=42,
    unit="km/day",
    statistic_type="mean",
    metric_denominator_n=186,
    denominator_definition="Respondents included in the main fair-pay summary table",
    observation_period_start="2022",
    observation_period_end="2023",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_summary",
    value_time_basis="day",
    distance_basis="source_reported_unspecified",
    extraction_notes=(
        "Distance definition does not establish paid-trip versus total work-related "
        "distance; project metric remains unmapped."
    ),
)

add_src029(
    source_locator="Policy brief pp. 3-4",
    source_metric_code="generic_working_hours",
    source_metric_label="Rerata waktu kerja per hari",
    source_value_text="11 hours/day",
    source_unit="hours/day",
    metric_family="activity",
    project_metric="working_hours_source_reported",
    mapping_status="mapped_with_caveat",
    value_numeric=11,
    unit="hours/day",
    statistic_type="mean",
    metric_denominator_n=186,
    denominator_definition="Respondents included in the main fair-pay summary table",
    observation_period_start="2022",
    observation_period_end="2023",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_summary",
    value_time_basis="day",
    working_time_basis="source_reported_unspecified",
    extraction_notes="Generic source working time is not relabelled as online or productive time.",
)

add_src029(
    source_locator="Policy brief p. 8",
    source_metric_code="driver_reported_deduction_rate_category",
    source_metric_label="Respondents reporting application deduction reaching 20 percent",
    source_value_text="52.9%",
    source_unit="percent",
    metric_family="deduction",
    value_numeric=52.9,
    unit="percent",
    statistic_type="share",
    category_label="20% deduction",
    category_lower_bound=20,
    category_upper_bound=20,
    category_bound_unit="percent",
    metric_denominator_n=225,
    denominator_definition="Full survey sample",
    observation_period_start="2023-04",
    observation_period_end="2023-05",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_distribution",
    extraction_notes="Driver-reported deduction; not transaction-level realized deduction.",
)

add_src029(
    source_locator="Policy brief p. 6",
    source_metric_code="generic_working_hours",
    source_metric_label="Respondents working 9-16 hours per day",
    source_value_text="68.9%",
    source_unit="percent",
    metric_family="activity",
    project_metric="working_hours_source_reported",
    mapping_status="mapped_with_caveat",
    value_numeric=68.9,
    unit="percent",
    statistic_type="share",
    category_label="9-16 hours/day",
    category_lower_bound=9,
    category_upper_bound=16,
    category_bound_unit="hours/day",
    metric_denominator_n=225,
    denominator_definition="Full survey sample",
    observation_period_start="2023-04",
    observation_period_end="2023-05",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_distribution",
    value_time_basis="day",
    working_time_basis="source_reported_unspecified",
    extraction_notes="Generic working time; not online/productive time.",
)

for category_label, lower, upper, share in [
    ("6-7 workdays/week", 6, 7, 79.6),
    ("7 workdays/week", 7, 7, 42.2),
]:
    add_src029(
        source_locator="Policy brief p. 6",
        source_metric_code="workdays_per_week",
        source_metric_label=category_label,
        source_value_text=f"{share}%",
        source_unit="percent",
        metric_family="activity",
        value_numeric=share,
        unit="percent",
        statistic_type="share",
        category_label=category_label,
        category_lower_bound=lower,
        category_upper_bound=upper,
        category_bound_unit="days/week",
        metric_denominator_n=225,
        denominator_definition="Full survey sample",
        observation_period_start="2023-04",
        observation_period_end="2023-05",
        temporal_evidence_status="contemporaneous",
        observation_level="sample_distribution",
        value_time_basis="week",
        extraction_notes="Source-specific workday metric retained without creating a new locked project metric.",
    )

add_src029(
    source_locator="Policy brief p. 6",
    source_metric_code="generic_working_hours",
    source_metric_label="Respondents working 61-112 hours per week",
    source_value_text="58.7%",
    source_unit="percent",
    metric_family="activity",
    project_metric="working_hours_source_reported",
    mapping_status="mapped_with_caveat",
    value_numeric=58.7,
    unit="percent",
    statistic_type="share",
    category_label="61-112 hours/week",
    category_lower_bound=61,
    category_upper_bound=112,
    category_bound_unit="hours/week",
    metric_denominator_n=225,
    denominator_definition="Full survey sample",
    observation_period_start="2023-04",
    observation_period_end="2023-05",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_distribution",
    value_time_basis="week",
    working_time_basis="source_reported_unspecified",
    extraction_notes="Generic working time; not online/productive time.",
)

src029_path = STANDARDIZED_DIR / "src029_ideas_2023_driver_observations.csv"
write_csv(src029_path, src029_rows, STANDARDIZED_COLUMNS)

print(f"Wrote {src029_path.relative_to(REPO_DIR)} with {len(src029_rows)} observations.")


Wrote data/standardized/src029_ideas_2023_driver_observations.csv with 14 observations.



## 7. Standardize SRC030 — IDEAS 2020 driver evidence

**Purpose:** retain only numerical observations whose source labels and denominators can be resolved sufficiently. The source-defined net-income category remains unmapped because its cost boundary is not sufficiently specified; work-time evidence is retained with item-specific denominators.


In [7]:

SRC030_META = {
    "sample_size": 542,
    "sample_population": "Ojol drivers in Jabodetabek; non-probability purposive sample",
    "geography": "Jabodetabek",
    "platform": "multi-platform",
    "service_type": "mixed",
    "vehicle_type": "motorcycle",
}

src030_rows = []

def add_src030(**kwargs):
    kwargs.setdefault("source_id", "SRC030")
    kwargs.setdefault("observation_id", f"SRC030-O{len(src030_rows)+1:03d}")
    for key, value in SRC030_META.items():
        kwargs.setdefault(key, value)
    src030_rows.append(build_row(**kwargs))

add_src030(
    source_locator="Ojol section p. 14",
    source_metric_code="source_defined_net_income_unspecified_cost_basis",
    source_metric_label="Pendapatan bersih rata-rata per hari below Rp100,000",
    source_value_text="54.2%",
    source_unit="percent",
    metric_family="earnings",
    value_numeric=54.2,
    unit="percent",
    statistic_type="share",
    category_label="< Rp100,000/day",
    category_lower_bound="",
    category_upper_bound=100000,
    category_bound_unit="IDR/day",
    metric_denominator_n=542,
    denominator_definition="Ojol respondent base for source-defined net daily income distribution",
    observation_period_start="2020",
    observation_period_end="2020",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_distribution",
    earnings_layer="source_defined_net",
    value_time_basis="day",
    cost_boundary="unknown",
    currency="IDR",
    nominal_real_status="nominal",
    extraction_notes="Source-defined 'net' cost boundary is not sufficiently specified for project net operating earnings.",
)

add_src030(
    source_locator="Ojol section p. 16",
    source_metric_code="generic_working_hours",
    source_metric_label="Respondents working more than 8 hours per day",
    source_value_text="77.4%",
    source_unit="percent",
    metric_family="activity",
    project_metric="working_hours_source_reported",
    mapping_status="mapped_with_caveat",
    value_numeric=77.4,
    unit="percent",
    statistic_type="share",
    category_label=">8 hours/day",
    category_lower_bound=8,
    category_upper_bound="",
    category_bound_unit="hours/day",
    metric_denominator_n=540,
    denominator_definition="Respondents in daily working-time figure",
    observation_period_start="2020",
    observation_period_end="2020",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_distribution",
    value_time_basis="day",
    working_time_basis="source_reported_unspecified",
    extraction_notes="Generic working time; not online/productive time.",
)

add_src030(
    source_locator="Ojol section p. 16",
    source_metric_code="generic_working_hours",
    source_metric_label="Respondents spending 12 hours working per day",
    source_value_text="28.3%",
    source_unit="percent",
    metric_family="activity",
    project_metric="working_hours_source_reported",
    mapping_status="mapped_with_caveat",
    value_numeric=28.3,
    unit="percent",
    statistic_type="share",
    category_label="12 hours/day",
    category_lower_bound=12,
    category_upper_bound=12,
    category_bound_unit="hours/day",
    metric_denominator_n=540,
    denominator_definition="Respondents in daily working-time figure",
    observation_period_start="2020",
    observation_period_end="2020",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_distribution",
    value_time_basis="day",
    working_time_basis="source_reported_unspecified",
    extraction_notes="Generic working time; not online/productive time.",
)

for category_label, lower, upper, share in [
    (">5 workdays/week", 5, "", 82.2),
    ("7 workdays/week", 7, 7, 49.3),
]:
    add_src030(
        source_locator="Ojol section p. 16",
        source_metric_code="workdays_per_week",
        source_metric_label=category_label,
        source_value_text=f"{share}%",
        source_unit="percent",
        metric_family="activity",
        value_numeric=share,
        unit="percent",
        statistic_type="share",
        category_label=category_label,
        category_lower_bound=lower,
        category_upper_bound=upper,
        category_bound_unit="days/week",
        metric_denominator_n=540,
        denominator_definition="Respondents in workdays-per-week figure",
        observation_period_start="2020",
        observation_period_end="2020",
        temporal_evidence_status="contemporaneous",
        observation_level="sample_distribution",
        value_time_basis="week",
        extraction_notes="Source-specific workday metric retained without creating a new locked project metric.",
    )

add_src030(
    source_locator="Ojol section p. 16",
    source_metric_code="generic_working_hours",
    source_metric_label="Respondents working more than 60 hours per week",
    source_value_text="73.0%",
    source_unit="percent",
    metric_family="activity",
    project_metric="working_hours_source_reported",
    mapping_status="mapped_with_caveat",
    value_numeric=73.0,
    unit="percent",
    statistic_type="share",
    category_label=">60 hours/week",
    category_lower_bound=60,
    category_upper_bound="",
    category_bound_unit="hours/week",
    metric_denominator_n=540,
    denominator_definition="Respondents in weekly working-time figure",
    observation_period_start="2020",
    observation_period_end="2020",
    temporal_evidence_status="contemporaneous",
    observation_level="sample_distribution",
    value_time_basis="week",
    working_time_basis="source_reported_unspecified",
    extraction_notes="Generic working time; not online/productive time.",
)

src030_path = STANDARDIZED_DIR / "src030_ideas_2020_driver_observations.csv"
write_csv(src030_path, src030_rows, STANDARDIZED_COLUMNS)

print(f"Wrote {src030_path.relative_to(REPO_DIR)} with {len(src030_rows)} observations.")


Wrote data/standardized/src030_ideas_2020_driver_observations.csv with 6 observations.



## 8. Standardize SRC017 — INDEF–PPPI driver survey context

**Purpose:** retain selected driver perceptions and preferences about promotions, platform choice, commission scenarios, service differences, and bonus experience as contextual survey evidence. These observations are not realized transaction deductions or causal economic effects.


In [8]:

SRC017_META = {
    "sample_size": 1000,
    "sample_population": "Ojol drivers in the INDEF-PPPI survey component; purposive sample",
    "geography": "Indonesia",
    "platform": "multi-platform",
    "service_type": "mixed",
    "vehicle_type": "motorcycle",
}

src017_rows = []

def add_src017(**kwargs):
    kwargs.setdefault("source_id", "SRC017")
    kwargs.setdefault("observation_id", f"SRC017-O{len(src017_rows)+1:03d}")
    for key, value in SRC017_META.items():
        kwargs.setdefault(key, value)
    src017_rows.append(build_row(**kwargs))

for category_label, share in [
    ("Perceived impact", 87.0),
    ("No perceived impact", 13.0),
]:
    add_src017(
        source_locator="Report p. 32, Figure 3.6",
        source_metric_code="promo_impact_perception",
        source_metric_label="Persepsi dampak promo terhadap kondisi kerja pengemudi",
        source_value_text=f"{share}%",
        source_unit="percent",
        metric_family="context",
        value_numeric=share,
        unit="percent",
        statistic_type="share",
        category_label=category_label,
        metric_denominator_n="",
        denominator_definition="Driver survey component; figure-specific n not stated in extracted text",
        observation_period_start="2025",
        observation_period_end="2025",
        temporal_evidence_status="reference_period",
        observation_level="sample_distribution",
        evidence_type="survey_perception",
        extraction_notes="Perception evidence; not an observed change in earnings, orders, or deductions.",
    )

for category_label, share in [
    ("More stable order volume", 60.0),
    ("Comfortable/familiar with app", 58.0),
    ("More attractive bonus/incentive", 31.0),
    ("Better protection/insurance", 27.0),
]:
    add_src017(
        source_locator="Report p. 39, Figure 3.12",
        source_metric_code="platform_loyalty_reason",
        source_metric_label="Reasons for driver loyalty to an application when commission is a consideration",
        source_value_text=f"{share}%",
        source_unit="percent",
        metric_family="context",
        value_numeric=share,
        unit="percent",
        statistic_type="multi_response_share",
        category_label=category_label,
        metric_denominator_n="",
        denominator_definition="Driver survey component; multiple-response item; figure-specific n not stated in extracted text",
        observation_period_start="2025",
        observation_period_end="2025",
        temporal_evidence_status="reference_period",
        observation_level="sample_distribution",
        evidence_type="survey_preference",
        extraction_notes="Preference/reason share; percentages are not mutually exclusive and do not measure realized commission effects.",
    )

add_src017(
    source_locator="Report p. 39, Figure 3.13",
    source_metric_code="commission_platform_preference",
    source_metric_label="Drivers preferring an app with 20 percent commission over an app with 10 percent commission",
    source_value_text="59%",
    source_unit="percent",
    metric_family="context",
    value_numeric=59.0,
    unit="percent",
    statistic_type="share",
    category_label="Prefers relatively higher 20% commission app",
    metric_denominator_n="",
    denominator_definition="Driver survey component; figure-specific n not stated in extracted text",
    observation_period_start="2025",
    observation_period_end="2025",
    temporal_evidence_status="reference_period",
    observation_level="sample_distribution",
    evidence_type="survey_preference",
    extraction_notes="Preference between stated commission scenarios; not a realized deduction observation.",
)

for category_label, share in [
    ("Perceives any difference in treatment/service", 79.0),
    ("Perceives significant difference", 32.0),
    ("Perceives less significant difference", 47.0),
    ("Perceives no difference", 17.0),
]:
    add_src017(
        source_locator="Report pp. 39-40, Figure 3.14",
        source_metric_code="commission_service_difference_perception",
        source_metric_label="Perceived difference in treatment/service between applications with different commission levels",
        source_value_text=f"{share}%",
        source_unit="percent",
        metric_family="context",
        value_numeric=share,
        unit="percent",
        statistic_type="share",
        category_label=category_label,
        metric_denominator_n="",
        denominator_definition="Driver survey component; exact item base not stated in extracted text",
        observation_period_start="2025",
        observation_period_end="2025",
        temporal_evidence_status="reference_period",
        observation_level="sample_distribution",
        evidence_type="survey_perception",
        extraction_notes="Perception evidence; categories are preserved as reported and no missing residual category is inferred.",
    )

for category_label, share in [
    ("Ever received incentive/bonus", 72.0),
    ("Never received incentive/bonus", 28.0),
]:
    add_src017(
        source_locator="Report p. 48, Figure 3.23",
        source_metric_code="bonus_receipt_experience",
        source_metric_label="Pengalaman pengemudi dalam menerima insentif atau bonus",
        source_value_text=f"{share}%",
        source_unit="percent",
        metric_family="incentive",
        value_numeric=share,
        unit="percent",
        statistic_type="share",
        category_label=category_label,
        metric_denominator_n="",
        denominator_definition="Driver survey component; figure-specific n not stated in extracted text",
        observation_period_start="2025",
        observation_period_end="2025",
        temporal_evidence_status="reference_period",
        observation_level="sample_distribution",
        evidence_type="survey_self_report",
        extraction_notes="Experience prevalence only; not a monetary incentive amount.",
    )

for category_label, share in [
    ("Access to promo/incentive", 69.0),
    ("Number of orders", 68.0),
    ("Platform communication", 16.0),
]:
    add_src017(
        source_locator="Report pp. 39-40, discussion following Figure 3.14",
        source_metric_code="perceived_difference_reason",
        source_metric_label="Reasons among drivers perceiving different treatment/service",
        source_value_text=f"{share}%",
        source_unit="percent",
        metric_family="context",
        value_numeric=share,
        unit="percent",
        statistic_type="multi_response_share",
        category_label=category_label,
        metric_denominator_n="",
        denominator_definition="Subgroup reporting a perceived treatment/service difference; exact subgroup n not stated in extracted text",
        observation_period_start="2025",
        observation_period_end="2025",
        temporal_evidence_status="reference_period",
        observation_level="subsample_distribution",
        evidence_type="survey_perception",
        extraction_notes="Subgroup perception; not an observed causal effect of commission.",
    )

src017_path = STANDARDIZED_DIR / "src017_indef_pppi_2025_driver_context_observations.csv"
write_csv(src017_path, src017_rows, STANDARDIZED_COLUMNS)

print(f"Wrote {src017_path.relative_to(REPO_DIR)} with {len(src017_rows)} observations.")


Wrote data/standardized/src017_indef_pppi_2025_driver_context_observations.csv with 16 observations.



## 9. Record Stage 2 reference inputs

**Purpose:** retain Stage 2D reference inputs in a dedicated registry without converting regulatory, fuel-price, CPI, or platform-policy evidence into driver-level observations. These references remain subject to period, geography, product, service, and evidence-layer matching.


In [9]:

reference_rows = [
    {
        "reference_record_id": "S2REF001",
        "source_id": "SRC015",
        "reference_type": "cpi",
        "reference_period_or_effective_date": "monthly 2025",
        "geography": "Indonesia and 38 provinces",
        "product_or_scope": "CPI 2022=100",
        "evidence_layer": "official_statistics",
        "stage2_role": "inflation_reference",
        "use_constraint": "Use only when observation geography and period are compatible; CPI is not a driver operating-cost observation.",
    },
    {
        "reference_record_id": "S2REF002",
        "source_id": "SRC016",
        "reference_type": "cpi",
        "reference_period_or_effective_date": "monthly 2025",
        "geography": "150 kabupaten/kota",
        "product_or_scope": "CPI 2022=100",
        "evidence_layer": "official_statistics",
        "stage2_role": "city_level_inflation_reference",
        "use_constraint": "Prefer geography-matched CPI where eligible; base-year discontinuities must be reconciled before cross-period real-value transformation.",
    },
    {
        "reference_record_id": "S2REF003",
        "source_id": "SRC022",
        "reference_type": "fuel_price",
        "reference_period_or_effective_date": "effective 2026-07-01",
        "geography": "Indonesia; product-specific regional prices",
        "product_or_scope": "motorcycle-compatible gasoline products",
        "evidence_layer": "secondary_with_official_company_lineage",
        "stage2_role": "fuel_scenario_reference",
        "use_constraint": "Fuel price must match product, effective date, and geography; it is not observed driver fuel expenditure.",
    },
    {
        "reference_record_id": "S2REF004",
        "source_id": "SRC028",
        "reference_type": "regulatory_rule",
        "reference_period_or_effective_date": "2022-08-04 until revoked",
        "geography": "Indonesia by tariff zones",
        "product_or_scope": "motorcycle passenger transport",
        "evidence_layer": "regulatory_deduction_rule",
        "stage2_role": "historical_normative_reference",
        "use_constraint": "Historical instrument only; regulatory fee ceiling and cost structure must not be substituted for realized driver deductions.",
    },
    {
        "reference_record_id": "S2REF005",
        "source_id": "SRC026",
        "reference_type": "platform_policy",
        "reference_period_or_effective_date": "implementation described from 2026-07-01",
        "geography": "Indonesia",
        "product_or_scope": "Grab two-wheel passenger services",
        "evidence_layer": "platform_stated_deduction",
        "stage2_role": "platform_policy_context",
        "use_constraint": "Company-stated implementation only; not transaction-level realized deduction evidence.",
    },
    {
        "reference_record_id": "S2REF006",
        "source_id": "SRC027",
        "reference_type": "platform_fee_definition",
        "reference_period_or_effective_date": "current at Stage 1 retrieval",
        "geography": "Indonesia",
        "product_or_scope": "GoSend",
        "evidence_layer": "platform_help_document",
        "stage2_role": "service_specific_fee_definition",
        "use_constraint": "GoSend definitions are service-specific and must not be generalized to GoRide, GoFood, or a universal driver commission.",
    },
]

for row in reference_rows:
    if row["source_id"] not in registered_source_ids:
        raise RuntimeError(f"Reference input source is absent from Stage 1 registry: {row['source_id']}")

reference_path = METADATA_DIR / "stage2_reference_input_registry.csv"
write_csv(
    reference_path,
    reference_rows,
    [
        "reference_record_id",
        "source_id",
        "reference_type",
        "reference_period_or_effective_date",
        "geography",
        "product_or_scope",
        "evidence_layer",
        "stage2_role",
        "use_constraint",
    ],
)

print(f"Wrote {reference_path.relative_to(REPO_DIR)} with {len(reference_rows)} reference records.")


Wrote metadata/stage2_reference_input_registry.csv with 6 reference records.



## 10. Record Stage 2 methodological decisions

**Purpose:** preserve the material standardization decisions that affect later comparability, calculation eligibility, and interpretation. These records describe research methodology and evidence boundaries.


In [10]:

decision_rows = [
    {
        "decision_id": "S2D001",
        "decision": "Preserve source-defined economic concepts before project mapping.",
        "rationale": "The same words such as income, net income, cost, distance, and working time can represent materially different economic concepts.",
        "evidence_basis": "Stage 0 concept gate; SRC010; SRC013; SRC029; SRC030",
        "analytical_implication": "Ambiguous source metrics remain source-defined rather than being forced into gross, receipts, net, total distance, or online-time metrics.",
        "intended_report_destination": "Methodology — measurement and standardization",
    },
    {
        "decision_id": "S2D002",
        "decision": "Use source_metric_code as a narrower semantic layer beneath the locked Stage 0 measurement dictionary.",
        "rationale": "Several valid source concepts do not have a safe one-to-one project metric and should not require reopening the locked Stage 0 dictionary.",
        "evidence_basis": "Stage 2B schema; Stage 0 measurement dictionary",
        "analytical_implication": "project_metric may remain blank while the observation stays usable for source-specific or contextual analysis.",
        "intended_report_destination": "Methodology — data architecture",
    },
    {
        "decision_id": "S2D003",
        "decision": "Add category_bound_unit as a Stage 2 implementation field.",
        "rationale": "Numeric bracket bounds require a unit that may differ from value_numeric, which is often a respondent percentage.",
        "evidence_basis": "SRC010 income brackets; SRC013 and SRC029 workload/deduction brackets",
        "analytical_implication": "Bracket values can be parsed without losing the original category label or confusing percent values with IDR/time/activity bounds.",
        "intended_report_destination": "Methodology — standardization schema",
    },
    {
        "decision_id": "S2D004",
        "decision": "Write one standardized observation file per source rather than a pooled Stage 2 dataset.",
        "rationale": "Source definitions, periods, samples, geographies, evidence layers, and denominators are not uniformly comparable.",
        "evidence_basis": "Stage 0 comparability framework; Stage 1 coverage validation",
        "analytical_implication": "Cross-source pooling is deferred until Stage 3 after Stage 2F comparability tagging and Stage 2G extraction validation.",
        "intended_report_destination": "Methodology — comparability and data flow",
    },
    {
        "decision_id": "S2D005",
        "decision": "Classify recalled historical earnings as retrospective_recall rather than independent historical waves.",
        "rationale": "SRC010 and SRC029 collect historical values retrospectively from respondents interviewed later.",
        "evidence_basis": "SRC010; SRC029",
        "analytical_implication": "Historical recall can support caveated context but is not treated as an independent panel or repeated cross-section.",
        "intended_report_destination": "Methodology — temporal evidence",
    },
    {
        "decision_id": "S2D006",
        "decision": "Keep fuel-plus-food/drink source bundles outside project operating-cost totals.",
        "rationale": "Food/drink is personal consumption rather than a driver operating cost under the locked project boundary.",
        "evidence_basis": "SRC013; SRC029; Stage 0 operating-cost rule",
        "analytical_implication": "Source-reported mixed cost values and ratios remain visible but cannot directly enter project net operating earnings.",
        "intended_report_destination": "Methodology — operating-cost boundary",
    },
    {
        "decision_id": "S2D007",
        "decision": "Keep driver-reported deduction categories separate from realized driver deduction rates.",
        "rationale": "Self-reported brackets do not establish matched transaction-level deduction amounts and gross earnings bases.",
        "evidence_basis": "SRC013; SRC029; Stage 0 deduction framework",
        "analytical_implication": "No realized_driver_deduction_rate is calculated from these survey percentages.",
        "intended_report_destination": "Methodology — deductions and regulatory evidence",
    },
    {
        "decision_id": "S2D008",
        "decision": "Leave source-reported daily distance unmapped when paid-trip versus total work-related distance is unresolved.",
        "rationale": "Fuel and vehicle-cost allocation depend materially on whether unpaid pickup and repositioning distance are included.",
        "evidence_basis": "SRC013; SRC029; Stage 0 distance rules",
        "analytical_implication": "Ambiguous distance cannot be used as total_work_related_distance_km without a later explicit scenario or stronger definition.",
        "intended_report_destination": "Methodology — workload and distance",
    },
    {
        "decision_id": "S2D009",
        "decision": "Map generic working time only to working_hours_source_reported.",
        "rationale": "The sources do not establish that generic work hours equal online or productive/engaged hours.",
        "evidence_basis": "SRC013; SRC029; SRC030; Stage 0 time rules",
        "analytical_implication": "Later hourly economics must retain the time-basis caveat and cannot silently substitute online/productive denominators.",
        "intended_report_destination": "Methodology — working-time measures",
    },
    {
        "decision_id": "S2D010",
        "decision": "Do not duplicate 2023 IDEAS quantitative observations from SRC013 when the canonical 2023 source SRC029 is available.",
        "rationale": "SRC013 reproduces comparison values from the earlier IDEAS survey, which would otherwise double count the same underlying evidence.",
        "evidence_basis": "SRC013; SRC029",
        "analytical_implication": "2023 observations are standardized under SRC029 only unless SRC013 provides a distinct 2025 comparison metric needed later.",
        "intended_report_destination": "Methodology — source lineage and duplicate evidence",
    },
    {
        "decision_id": "S2D011",
        "decision": "Treat INDEF-PPPI promotion, commission, platform-choice, and service-difference results as perception/preference context unless an observation directly measures economics.",
        "rationale": "Survey perceptions and stated choices do not establish realized deductions or causal changes in earnings and orders.",
        "evidence_basis": "SRC017",
        "analytical_implication": "SRC017 context observations are retained but excluded from realized deduction and driver unit-economics calculations.",
        "intended_report_destination": "Methodology — evidence layers",
    },
    {
        "decision_id": "S2D012",
        "decision": "Retain SRC020 as supporting association evidence without Stage 2 numerical observations.",
        "rationale": "The article metadata and abstract establish study scope and association findings but do not provide sufficiently defined numerical tables for project metric standardization.",
        "evidence_basis": "SRC020; Stage 1 source assessment",
        "analytical_implication": "SRC020 may inform later discussion of associations with appropriate scope limits but supplies no standardized numeric unit-economics rows in Stage 2E.",
        "intended_report_destination": "Methodology — evidence coverage limitations",
    },
    {
        "decision_id": "S2D013",
        "decision": "Retain only SRC030 numerical values whose labels and item denominators can be resolved sufficiently.",
        "rationale": "The 2020 brief contains several chart-based earnings categories with source-defined net terminology whose full cost basis and some category text are not sufficiently explicit for stronger mapping.",
        "evidence_basis": "SRC030",
        "analytical_implication": "Work-time prevalence and one clearly described source-defined net-income category are standardized; unresolved chart categories are not reconstructed.",
        "intended_report_destination": "Methodology — extraction limitations",
    },
    {
        "decision_id": "S2D014",
        "decision": "Use SRC017 as the canonical numerical source for the INDEF-PPPI survey programme and do not separately standardize overlapping SRC025 presentation figures by default.",
        "rationale": "The report and presentation describe the same underlying research programme and separately entering overlapping figures would duplicate evidence.",
        "evidence_basis": "SRC017; SRC025",
        "analytical_implication": "SRC025 remains a companion reference and may be used only when it contributes a distinct, non-duplicative observation.",
        "intended_report_destination": "Methodology — source lineage and duplicate evidence",
    },
]

decision_path = METADATA_DIR / "stage2_methodological_decision_log.csv"
write_csv(
    decision_path,
    decision_rows,
    [
        "decision_id",
        "decision",
        "rationale",
        "evidence_basis",
        "analytical_implication",
        "intended_report_destination",
    ],
)

print(f"Wrote {decision_path.relative_to(REPO_DIR)} with {len(decision_rows)} decisions.")


Wrote metadata/stage2_methodological_decision_log.csv with 14 decisions.



## 11. Build the Stage 2E output manifest and run standardization checks

**Purpose:** register the source-specific Stage 2E outputs and run implementation checks for unique observations, valid source IDs, locked project-metric mappings, percentage ranges, mixed-cost boundaries, recall classification, and the absence of Stage 2E comparability assignments or analytical transformations.

These are Stage 2E implementation checks; the full extraction-validation gate is completed in Stage 2G.


In [11]:

standardized_outputs = [
    {
        "source_id": "SRC010",
        "output_path": "data/standardized/src010_polling_institute_2022_driver_observations.csv",
        "output_role": "primary_quantitative",
        "observation_count": len(src010_rows),
        "stage2e_status": "standardized",
        "key_scope_note": "Daily income layer remains unspecified; historical values are retrospective recall.",
    },
    {
        "source_id": "SRC013",
        "output_path": "data/standardized/src013_ideas_2025_driver_observations.csv",
        "output_role": "primary_quantitative",
        "observation_count": len(src013_rows),
        "stage2e_status": "standardized_with_semantic_caveats",
        "key_scope_note": "Mixed fuel-food cost bundle, driver-reported deductions, ambiguous distance, and 62/67 locality discrepancy remain explicit.",
    },
    {
        "source_id": "SRC017",
        "output_path": "data/standardized/src017_indef_pppi_2025_driver_context_observations.csv",
        "output_role": "contextual_survey",
        "observation_count": len(src017_rows),
        "stage2e_status": "standardized_context_only",
        "key_scope_note": "Perception/preference evidence is not realized deduction or causal unit-economics evidence.",
    },
    {
        "source_id": "SRC029",
        "output_path": "data/standardized/src029_ideas_2023_driver_observations.csv",
        "output_role": "primary_quantitative",
        "observation_count": len(src029_rows),
        "stage2e_status": "standardized_with_temporal_caveats",
        "key_scope_note": "Retrospective earnings recall is separated from the contemporaneous 2022-2023 sample summary.",
    },
    {
        "source_id": "SRC030",
        "output_path": "data/standardized/src030_ideas_2020_driver_observations.csv",
        "output_role": "primary_quantitative",
        "observation_count": len(src030_rows),
        "stage2e_status": "standardized_with_definition_caveats",
        "key_scope_note": "Only sufficiently resolved numerical observations are retained; source-defined net remains unmapped.",
    },
    {
        "source_id": "SRC020",
        "output_path": "",
        "output_role": "supporting_association",
        "observation_count": 0,
        "stage2e_status": "retained_without_numeric_rows",
        "key_scope_note": "Study scope and association evidence retained; no sufficiently defined numerical table values entered in Stage 2E.",
    },
    {
        "source_id": "SRC025",
        "output_path": "",
        "output_role": "companion_same_program",
        "observation_count": 0,
        "stage2e_status": "not_separately_standardized_same_program",
        "key_scope_note": "Companion presentation to SRC017; overlapping figures are not duplicated.",
    },
]

manifest_path = METADATA_DIR / "stage2_standardized_output_manifest.csv"
write_csv(
    manifest_path,
    standardized_outputs,
    [
        "source_id",
        "output_path",
        "output_role",
        "observation_count",
        "stage2e_status",
        "key_scope_note",
    ],
)

all_rows = src010_rows + src013_rows + src017_rows + src029_rows + src030_rows

# 1. Unique observation IDs.
observation_ids = [row["observation_id"] for row in all_rows]
duplicates = [key for key, count in Counter(observation_ids).items() if count > 1]
if duplicates:
    raise RuntimeError(f"Duplicate observation IDs: {duplicates}")

# 2. Every source is registered.
unregistered_sources = sorted(
    {row["source_id"] for row in all_rows if row["source_id"] not in registered_source_ids}
)
if unregistered_sources:
    raise RuntimeError(f"Unregistered source IDs in standardized observations: {unregistered_sources}")

# 3. Every mapped project metric exists in the locked Stage 0 dictionary.
invalid_project_metrics = sorted(
    {
        row["project_metric"]
        for row in all_rows
        if row["project_metric"] and row["project_metric"] not in project_metrics
    }
)
if invalid_project_metrics:
    raise RuntimeError(f"Invalid locked project metrics: {invalid_project_metrics}")

# 4. Percent values are bounded.
invalid_percent_rows = [
    row["observation_id"]
    for row in all_rows
    if row["unit"] == "percent"
    and not (0 <= float(row["value_numeric"]) <= 100)
]
if invalid_percent_rows:
    raise RuntimeError(f"Percent values outside [0, 100]: {invalid_percent_rows}")

# 5. Mixed fuel-food bundles cannot map to a project cost metric.
invalid_mixed_cost = [
    row["observation_id"]
    for row in all_rows
    if row["source_metric_code"] in {
        "fuel_food_daily_cost_bundle",
        "fuel_food_cost_share_of_gross",
    }
    and (
        row["cost_boundary"] != "mixed_work_personal"
        or row["project_metric"] != ""
    )
]
if invalid_mixed_cost:
    raise RuntimeError(f"Mixed cost boundary violation: {invalid_mixed_cost}")

# 6. Recalled historical gross earnings remain marked as recall.
invalid_recall = [
    row["observation_id"]
    for row in all_rows
    if row["source_metric_code"] == "explicit_gross_driver_earnings_recalled"
    and row["temporal_evidence_status"] != "retrospective_recall"
]
if invalid_recall:
    raise RuntimeError(f"Historical recall classification violation: {invalid_recall}")

# 7. Generic working hours can map only to working_hours_source_reported.
invalid_hours = [
    row["observation_id"]
    for row in all_rows
    if row["source_metric_code"] == "generic_working_hours"
    and row["project_metric"] != "working_hours_source_reported"
]
if invalid_hours:
    raise RuntimeError(f"Working-time semantic mapping violation: {invalid_hours}")

# 8. Stage 2E performs no analytical transformation and assigns no cross-source comparability status.
transformed = [
    row["observation_id"]
    for row in all_rows
    if row["transformation_applied"] != "none"
    or row["transformation_formula"] not in ("", None)
]
if transformed:
    raise RuntimeError(f"Unexpected Stage 2E transformation: {transformed}")

premature_comparability = [
    row["observation_id"]
    for row in all_rows
    if row["comparability_status"] not in ("", None)
    or row["comparability_use"] not in ("", None)
]
if premature_comparability:
    raise RuntimeError(
        f"Cross-source comparability was assigned before Stage 2F: {premature_comparability}"
    )

# 9. No pooled Stage 2 analytical file is produced.
pooled_candidates = [
    p.name.lower()
    for p in STANDARDIZED_DIR.glob("*.csv")
    if any(token in p.name.lower() for token in ["combined", "pooled", "all_sources", "analysis_ready"])
]
if pooled_candidates:
    raise RuntimeError(f"Unexpected pooled Stage 2 file(s): {pooled_candidates}")

counts_by_source = Counter(row["source_id"] for row in all_rows)
counts_by_mapping = Counter(row["mapping_status"] for row in all_rows)

print("Stage 2E implementation checks passed.")
print(f"Total standardized observations: {len(all_rows)}")
print("\nObservations by source:")
for source_id in sorted(counts_by_source):
    print(f"- {source_id}: {counts_by_source[source_id]}")

print("\nMapping-status counts:")
for status, count in sorted(counts_by_mapping.items()):
    print(f"- {status}: {count}")

print("\nStage 2E files:")
for path in [
    schema_path,
    mapping_path,
    src010_path,
    src013_path,
    src017_path,
    src029_path,
    src030_path,
    reference_path,
    decision_path,
    manifest_path,
]:
    print(f"- {path.relative_to(REPO_DIR)}")


Stage 2E implementation checks passed.
Total standardized observations: 72

Observations by source:
- SRC010: 18
- SRC013: 18
- SRC017: 16
- SRC029: 14
- SRC030: 6

Mapping-status counts:
- mapped_exact: 3
- mapped_with_caveat: 7
- mapped_with_source_category: 3
- mapped_with_temporal_caveat: 3
- source_defined_unmapped: 56

Stage 2E files:
- metadata/stage2_extraction_schema.csv
- metadata/stage2_metric_mapping.csv
- data/standardized/src010_polling_institute_2022_driver_observations.csv
- data/standardized/src013_ideas_2025_driver_observations.csv
- data/standardized/src017_indef_pppi_2025_driver_context_observations.csv
- data/standardized/src029_ideas_2023_driver_observations.csv
- data/standardized/src030_ideas_2020_driver_observations.csv
- metadata/stage2_reference_input_registry.csv
- metadata/stage2_methodological_decision_log.csv
- metadata/stage2_standardized_output_manifest.csv



# Stage 2F — Cross-Source Comparability Tagging

Stage 2F evaluates whether standardized observations may be compared for a **specific analytical use**.

Comparability is not assigned as a permanent property of an observation. The same observation may be usable for one comparison and ineligible for another. Therefore Stage 2F keeps the Stage 2E source-specific files unchanged and writes a separate use-specific comparability assessment.

Allowed statuses remain:

- `directly_comparable`
- `comparable_with_transformation`
- `comparable_with_caveat`
- `context_only`
- `not_comparable`



## 12. Load and verify Stage 2E standardized observations

**Purpose:** reload the source-specific standardized files, verify unique observation identifiers, and establish the observation lookup used by the Stage 2F comparability assessment.


In [12]:

standardized_files = sorted(STANDARDIZED_DIR.glob("src*_*.csv"))

if not standardized_files:
    raise FileNotFoundError(
        f"No Stage 2E standardized CSV files were found in {STANDARDIZED_DIR}."
    )

stage2e_rows = []
for path in standardized_files:
    stage2e_rows.extend(read_csv(path))

observation_lookup = {
    row["observation_id"]: row
    for row in stage2e_rows
}

if len(stage2e_rows) != len(observation_lookup):
    raise RuntimeError(
        "Stage 2F cannot proceed because Stage 2E observation IDs are not unique."
    )

expected_source_counts = {
    "SRC010": 18,
    "SRC013": 18,
    "SRC017": 16,
    "SRC029": 14,
    "SRC030": 6,
}

actual_source_counts = Counter(
    row["source_id"] for row in stage2e_rows
)

if actual_source_counts != Counter(expected_source_counts):
    raise RuntimeError(
        "Stage 2E source counts do not match the validated Stage 2E baseline. "
        f"Expected {expected_source_counts}; found {dict(actual_source_counts)}."
    )

print("Stage 2E input verification passed.")
print(f"Standardized observations available: {len(stage2e_rows)}")
for source_id in sorted(actual_source_counts):
    print(f"- {source_id}: {actual_source_counts[source_id]}")


Stage 2E input verification passed.
Standardized observations available: 72
- SRC010: 18
- SRC013: 18
- SRC017: 16
- SRC029: 14
- SRC030: 6



## 13. Build use-specific comparability assessments

**Purpose:** assess the principal comparisons that later analysis may attempt. Each assessment identifies the relevant observations, transformation requirements, caveats, allowed use, and prohibited interpretation.

No source observation is modified and no cross-source statistic is calculated at this stage.


In [13]:

COMPARABILITY_COLUMNS = [
    "comparison_id",
    "comparison_use",
    "metric_concept",
    "source_ids",
    "observation_ids",
    "comparability_status",
    "required_transformation",
    "critical_caveats",
    "allowed_use",
    "prohibited_use",
    "rationale",
]

comparability_rows = []

def add_comparability(
    comparison_id,
    comparison_use,
    metric_concept,
    observation_ids,
    comparability_status,
    required_transformation,
    critical_caveats,
    allowed_use,
    prohibited_use,
    rationale,
):
    missing = [
        observation_id
        for observation_id in observation_ids
        if observation_id not in observation_lookup
    ]
    if missing:
        raise KeyError(
            f"{comparison_id} references missing Stage 2E observation IDs: {missing}"
        )

    source_ids = sorted(
        {
            observation_lookup[observation_id]["source_id"]
            for observation_id in observation_ids
        }
    )

    comparability_rows.append({
        "comparison_id": comparison_id,
        "comparison_use": comparison_use,
        "metric_concept": metric_concept,
        "source_ids": ";".join(source_ids),
        "observation_ids": ";".join(observation_ids),
        "comparability_status": comparability_status,
        "required_transformation": required_transformation,
        "critical_caveats": critical_caveats,
        "allowed_use": allowed_use,
        "prohibited_use": prohibited_use,
        "rationale": rationale,
    })


add_comparability(
    "S2C001",
    "cross_period_gross_daily_earnings",
    "explicit gross driver earnings",
    ["SRC029-O001", "SRC013-O001"],
    "comparable_with_transformation",
    "Inflation-adjust nominal IDR/day to a common price period using geographically compatible CPI where defensible.",
    "The 2023 observation is a Jabodetabek purposive sample; the 2025 observation has broader purposive coverage with an internal 62/67-locality discrepancy. Sampling frames and geography differ.",
    "Caveated comparison of source-sample gross daily earnings after documented inflation treatment.",
    "National trend inference; direct welfare conclusion; unadjusted nominal level comparison.",
    "Economic concept and daily time basis align, but monetary periods differ and sample/geography are not equivalent.",
)

add_comparability(
    "S2C002",
    "within_src029_recalled_gross_trajectory",
    "recalled explicit gross driver earnings",
    ["SRC029-O002", "SRC029-O003", "SRC029-O004"],
    "comparable_with_transformation",
    "Inflation-adjust recalled nominal IDR/day values to a common price period if level comparison is performed.",
    "All values are retrospective recall from longer-tenure respondents interviewed in 2023; they are not independent historical survey waves.",
    "Caveated within-source recalled trajectory or sensitivity context.",
    "Treating 2018-2019, 2020-2021, and 2022-2023 as repeated cross-sections or panel waves.",
    "Source question and subgroup lineage are common, but recall and nominal monetary periods prevent direct historical-wave comparison.",
)

add_comparability(
    "S2C003",
    "within_src010_income_distribution_over_time",
    "source-defined daily income distribution",
    [
        "SRC010-O001", "SRC010-O002", "SRC010-O003", "SRC010-O004",
        "SRC010-O005", "SRC010-O006", "SRC010-O007", "SRC010-O008",
        "SRC010-O009", "SRC010-O010", "SRC010-O011", "SRC010-O012",
        "SRC010-O013", "SRC010-O014", "SRC010-O015", "SRC010-O016",
        "SRC010-O017", "SRC010-O018",
    ],
    "comparable_with_caveat",
    "None in Stage 2; nominal bracket thresholds cannot be converted to real-income brackets without microdata.",
    "Income layer is unspecified; pre-pandemic and pandemic values are retrospective recall; fixed nominal brackets do not represent constant purchasing-power thresholds.",
    "Within-source comparison of reported nominal income-category distributions with explicit recall and bracket caveats.",
    "Mapping to gross/net earnings; real-income distribution comparison; independent-wave interpretation.",
    "The category structure is consistent within SRC010, but semantic and temporal limitations remain material.",
)

add_comparability(
    "S2C004",
    "src010_vs_explicit_gross_earnings",
    "daily income versus explicit gross earnings",
    [
        "SRC010-O013", "SRC010-O014", "SRC010-O015", "SRC010-O016",
        "SRC013-O001", "SRC029-O001",
    ],
    "not_comparable",
    "Not applicable.",
    "SRC010 income layer is unspecified and reported as a distribution; IDEAS observations are explicit gross means.",
    "Separate source-specific description only.",
    "Numerical gross-earnings comparison or pooled estimate.",
    "Economic layer and statistic type do not match.",
)

add_comparability(
    "S2C005",
    "cross_period_mixed_daily_cost_amount",
    "fuel plus food/drink daily cost bundle",
    ["SRC029-O005", "SRC013-O002"],
    "comparable_with_transformation",
    "Inflation-adjust nominal IDR/day to a common price period if comparing amounts.",
    "The bundle includes food/drink, which is outside project operating costs; geography and purposive sample coverage differ.",
    "Caveated comparison of the source-defined mixed bundle after inflation treatment.",
    "Treating the bundle as project operating cost or using it directly for project net operating earnings.",
    "Source-defined bundle and daily mean basis are similar, but monetary period and sample/geography require treatment and caveats.",
)

add_comparability(
    "S2C006",
    "cross_period_mixed_cost_share",
    "fuel plus food/drink share of gross earnings",
    ["SRC029-O006", "SRC013-O003"],
    "comparable_with_caveat",
    "None required for the reported percentage itself.",
    "The source-defined bundle includes personal food/drink; geography and purposive sample coverage differ.",
    "Caveated comparison of the source-defined bundle-to-gross ratio.",
    "Calling the percentage project operating-cost share or applying it outside the source-defined bundle.",
    "The ratio concept is aligned within the IDEAS source family, but cost-boundary and sample differences remain.",
)

add_comparability(
    "S2C007",
    "seven_day_workweek_prevalence",
    "share reporting exactly 7 workdays/week",
    ["SRC030-O005", "SRC029-O013", "SRC013-O018"],
    "comparable_with_caveat",
    "None.",
    "The 2020 and 2023 observations are Jabodetabek purposive samples; 2025 has broader purposive coverage. Survey periods and respondent composition differ.",
    "Sample-level descriptive comparison of exact seven-day workweek prevalence.",
    "National prevalence trend or causal interpretation.",
    "Category, time basis, unit, vehicle universe, and broad work concept align, but sample and geography do not.",
)

add_comparability(
    "S2C008",
    "reported_twenty_percent_deduction_prevalence",
    "share reporting a 20% application deduction",
    ["SRC029-O010", "SRC013-O005"],
    "comparable_with_caveat",
    "None.",
    "Driver-reported categories are not transaction-level realized deductions; geography, period, and purposive samples differ.",
    "Sample-level comparison of reported 20% deduction prevalence.",
    "Realized commission rate comparison; legal-compliance conclusion; transaction-level deduction estimate.",
    "The reported category is similar across the two IDEAS surveys, but evidence layer and sample scope constrain interpretation.",
)

add_comparability(
    "S2C009",
    "working_hours_mean_vs_distribution",
    "source-reported working time",
    ["SRC029-O009", "SRC013-O017", "SRC030-O002", "SRC030-O003"],
    "not_comparable",
    "Not applicable.",
    "One source reports a mean while others report nonmatching distribution thresholds; all are generic source-reported working time rather than proven online/productive hours.",
    "Separate source-specific description.",
    "Pooled mean working hours or direct prevalence comparison across nonmatching thresholds.",
    "Statistic type and threshold definitions differ materially.",
)

add_comparability(
    "S2C010",
    "working_hours_distribution_context",
    "long-working-time prevalence",
    ["SRC030-O002", "SRC030-O003", "SRC029-O011", "SRC029-O014", "SRC013-O017"],
    "context_only",
    "None.",
    "Thresholds differ (>8, 12, 9-16, 61-112/week, 9-12/day); daily and weekly bases differ; samples and geographies differ.",
    "Qualitative context that long source-reported working-time schedules appear in multiple studies.",
    "Exact cross-source prevalence ranking or pooled percentage.",
    "The observations address related workload concepts but are not numerically aligned.",
)

add_comparability(
    "S2C011",
    "orders_per_day_cross_source",
    "completed orders per day",
    ["SRC029-O007", "SRC013-O011", "SRC013-O012", "SRC013-O013"],
    "not_comparable",
    "Not applicable.",
    "SRC029 reports a mean; SRC013 reports selected brackets and does not provide complete microdata for mean reconstruction.",
    "Separate source-specific workload description.",
    "Direct mean comparison or reconstructed mean from incomplete bracket shares.",
    "Statistic types and available distribution detail are incompatible.",
)

add_comparability(
    "S2C012",
    "distance_per_day_cross_source",
    "source-reported daily distance",
    ["SRC029-O008", "SRC013-O014", "SRC013-O015", "SRC013-O016"],
    "not_comparable",
    "Not applicable.",
    "SRC029 reports a mean; SRC013 reports selected brackets; distance basis is unresolved as paid-trip versus total work-related distance.",
    "Separate source-specific distance description with basis caveat.",
    "Fuel-cost allocation using these values as total work distance; direct mean comparison.",
    "Statistic type differs and the distance concept is not sufficiently resolved.",
)

add_comparability(
    "S2C013",
    "source_defined_net_income_cross_source",
    "source-defined net income",
    ["SRC030-O001", "SRC013-O004"],
    "not_comparable",
    "Not applicable.",
    "SRC030 cost basis is unresolved; SRC013 net subtracts a mixed fuel-plus-food/drink bundle; one is a daily distribution and the other a monthly source estimate.",
    "Separate description of each source-defined net concept.",
    "Project net operating earnings comparison or time-basis conversion as if definitions matched.",
    "Cost boundaries, statistic types, and time bases are materially different.",
)

add_comparability(
    "S2C014",
    "bonus_evidence_cross_source",
    "bonus/incentive availability and receipt experience",
    ["SRC013-O007", "SRC013-O008", "SRC013-O009", "SRC013-O010", "SRC017-O012", "SRC017-O013"],
    "not_comparable",
    "Not applicable.",
    "SRC013 measures program availability and receipt frequency; SRC017 measures ever/never receipt experience. Denominators also differ or are unresolved.",
    "Separate source-specific incentive context.",
    "Direct bonus-prevalence comparison or monetary incentive inference.",
    "The survey questions measure different incentive concepts.",
)

add_comparability(
    "S2C015",
    "commission_perception_vs_reported_deduction",
    "commission preference/perception versus driver-reported deduction",
    ["SRC017-O007", "SRC017-O008", "SRC017-O009", "SRC017-O010", "SRC017-O011", "SRC029-O010", "SRC013-O005", "SRC013-O006"],
    "context_only",
    "None.",
    "SRC017 contains stated preference/perception; SRC013/SRC029 contain driver-reported deduction categories; none are matched transaction evidence.",
    "Contextual comparison of evidence layers while keeping them separate.",
    "Converting preferences or perceptions into realized deduction estimates.",
    "Related commission topics exist, but evidence types are not interchangeable.",
)

add_comparability(
    "S2C016",
    "project_net_operating_earnings_reconstruction",
    "gross earnings plus available cost evidence",
    ["SRC029-O001", "SRC029-O005", "SRC013-O001", "SRC013-O002", "SRC013-O004"],
    "not_comparable",
    "Not applicable in Stage 2.",
    "Available cost observations are mixed fuel-plus-food/drink bundles and the same-observation chain does not include complete operating costs and realized deductions.",
    "Evidence-gap documentation only.",
    "Constructing project net operating earnings from these components.",
    "The project accounting chain is incomplete and the cost boundary is incompatible with project net operating earnings.",
)

add_comparability(
    "S2C017",
    "regulatory_or_platform_policy_vs_realized_deduction",
    "normative/platform policy versus driver deduction evidence",
    ["SRC029-O010", "SRC013-O005", "SRC013-O006"],
    "context_only",
    "Scope and effective-date matching would be required in later regulatory reconciliation.",
    "Standardized driver evidence is self-reported; Stage 2 reference inputs are normative/platform-stated rather than matched realized transactions.",
    "Layered regulatory context and later reconciliation with explicit evidence labels.",
    "Legal compliance conclusion or assumption that ceilings/stated rates equal realized deductions.",
    "Normative, company-stated, and driver-reported evidence answer different questions.",
)

comparability_path = METADATA_DIR / "stage2_comparability_assessment.csv"
write_csv(
    comparability_path,
    comparability_rows,
    COMPARABILITY_COLUMNS,
)

print(
    f"Wrote {comparability_path.relative_to(REPO_DIR)} "
    f"with {len(comparability_rows)} use-specific assessments."
)


Wrote metadata/stage2_comparability_assessment.csv with 17 use-specific assessments.



## 14. Record Stage 2F methodological decisions

**Purpose:** add the decisions required to keep comparability use-specific, preserve inferential limits, and prevent monetary transformations or evidence-layer reconciliation from becoming implicit.


In [14]:

stage2f_decisions = [
    {
        "decision_id": "S2D015",
        "decision": "Store comparability as a use-specific assessment rather than a permanent observation-level property.",
        "rationale": "The same observation may be eligible for one analytical use and ineligible for another.",
        "evidence_basis": "Stage 0 comparability framework; Stage 2B architecture; Stage 2F assessments",
        "analytical_implication": "Stage 2E standardized rows remain unchanged; later analysis must join comparability by intended use.",
        "intended_report_destination": "Methodology — comparability framework",
    },
    {
        "decision_id": "S2D016",
        "decision": "Require documented inflation treatment before eligible cross-period nominal monetary level comparisons.",
        "rationale": "Nominal IDR values from different periods do not represent the same purchasing-power basis.",
        "evidence_basis": "SRC015; SRC016; Stage 0 inflation rule; S2C001; S2C002; S2C005",
        "analytical_implication": "Comparable-with-transformation monetary assessments cannot be analyzed as real level changes until Stage 3 applies an eligible CPI transformation.",
        "intended_report_destination": "Methodology — inflation and monetary comparability",
    },
    {
        "decision_id": "S2D017",
        "decision": "Do not interpret cross-source sample comparability as national representativeness.",
        "rationale": "Several surveys are purposive and differ in geography, recruitment, respondent composition, and period.",
        "evidence_basis": "SRC013; SRC017; SRC029; SRC030; Stage 1 sampling assessment",
        "analytical_implication": "Even aligned categories such as seven-day workweeks and reported 20 percent deductions support sample-level comparisons only.",
        "intended_report_destination": "Methodology — sampling and inference",
    },
    {
        "decision_id": "S2D018",
        "decision": "Keep normative, platform-stated, perception, driver-reported, and realized deduction evidence separate during comparability assessment.",
        "rationale": "These evidence layers answer different questions and cannot be substituted for one another.",
        "evidence_basis": "Stage 0 deduction framework; SRC013; SRC017; SRC026; SRC028; SRC029",
        "analytical_implication": "Regulatory or platform policy may contextualize driver evidence but cannot establish realized deduction or legal compliance without matched evidence.",
        "intended_report_destination": "Methodology — deduction and regulatory evidence layers",
    },
]

decision_rows_existing = read_csv(
    METADATA_DIR / "stage2_methodological_decision_log.csv"
)

existing_decision_ids = {
    row["decision_id"] for row in decision_rows_existing
}

duplicate_stage2f_ids = sorted(
    {
        row["decision_id"]
        for row in stage2f_decisions
        if row["decision_id"] in existing_decision_ids
    }
)

if duplicate_stage2f_ids:
    # Idempotent rerun: replace any prior Stage 2F decision rows with the
    # current locked definitions rather than duplicating them.
    stage2f_id_set = {
        row["decision_id"] for row in stage2f_decisions
    }
    decision_rows_existing = [
        row for row in decision_rows_existing
        if row["decision_id"] not in stage2f_id_set
    ]

decision_rows_updated = decision_rows_existing + stage2f_decisions

write_csv(
    METADATA_DIR / "stage2_methodological_decision_log.csv",
    decision_rows_updated,
    [
        "decision_id",
        "decision",
        "rationale",
        "evidence_basis",
        "analytical_implication",
        "intended_report_destination",
    ],
)

print(
    "Stage 2 methodological decision log updated: "
    f"{len(decision_rows_updated)} total decisions."
)


Stage 2 methodological decision log updated: 18 total decisions.



## 15. Validate Stage 2F comparability tagging

**Purpose:** verify valid status vocabulary, unique comparison identifiers, observation traceability, required transformation logic, preservation of blank Stage 2E observation-level comparability fields, and the absence of any unsupported directly-comparable classification.


In [15]:

allowed_comparability_statuses = {
    "directly_comparable",
    "comparable_with_transformation",
    "comparable_with_caveat",
    "context_only",
    "not_comparable",
}

comparison_ids = [
    row["comparison_id"] for row in comparability_rows
]

if len(comparison_ids) != len(set(comparison_ids)):
    raise RuntimeError("Stage 2F comparison IDs are not unique.")

invalid_statuses = sorted(
    {
        row["comparability_status"]
        for row in comparability_rows
        if row["comparability_status"] not in allowed_comparability_statuses
    }
)
if invalid_statuses:
    raise RuntimeError(
        f"Invalid Stage 2F comparability statuses: {invalid_statuses}"
    )

unresolved_observation_ids = []
for row in comparability_rows:
    for observation_id in row["observation_ids"].split(";"):
        if observation_id not in observation_lookup:
            unresolved_observation_ids.append(
                (row["comparison_id"], observation_id)
            )

if unresolved_observation_ids:
    raise RuntimeError(
        "Comparability assessment contains unresolved observation IDs: "
        f"{unresolved_observation_ids}"
    )

missing_transformations = [
    row["comparison_id"]
    for row in comparability_rows
    if row["comparability_status"] == "comparable_with_transformation"
    and row["required_transformation"].strip().lower() in {"", "none", "not applicable."}
]
if missing_transformations:
    raise RuntimeError(
        "Comparable-with-transformation assessments lack a documented "
        f"transformation: {missing_transformations}"
    )

premature_stage2e_tags = [
    row["observation_id"]
    for row in stage2e_rows
    if row["comparability_use"] not in ("", None)
    or row["comparability_status"] not in ("", None)
]
if premature_stage2e_tags:
    raise RuntimeError(
        "Stage 2E source rows were unexpectedly modified with global "
        f"comparability tags: {premature_stage2e_tags}"
    )

direct_comparisons = [
    row["comparison_id"]
    for row in comparability_rows
    if row["comparability_status"] == "directly_comparable"
]

status_counts = Counter(
    row["comparability_status"]
    for row in comparability_rows
)

summary_rows = [
    {
        "stage2f_status": "PASS_WITH_CAVEAT",
        "assessment_count": len(comparability_rows),
        "directly_comparable_count": status_counts.get("directly_comparable", 0),
        "comparable_with_transformation_count": status_counts.get("comparable_with_transformation", 0),
        "comparable_with_caveat_count": status_counts.get("comparable_with_caveat", 0),
        "context_only_count": status_counts.get("context_only", 0),
        "not_comparable_count": status_counts.get("not_comparable", 0),
        "key_conclusion": (
            "No assessed cross-source use is directly comparable without "
            "transformation, caveat, contextual restriction, or exclusion."
        ),
    }
]

summary_path = METADATA_DIR / "stage2_comparability_summary.csv"
write_csv(
    summary_path,
    summary_rows,
    [
        "stage2f_status",
        "assessment_count",
        "directly_comparable_count",
        "comparable_with_transformation_count",
        "comparable_with_caveat_count",
        "context_only_count",
        "not_comparable_count",
        "key_conclusion",
    ],
)

print("Stage 2F validation passed.")
print(f"Overall status: {summary_rows[0]['stage2f_status']}")
print(f"Comparability assessments: {len(comparability_rows)}")
print("Status counts:")
for status in [
    "directly_comparable",
    "comparable_with_transformation",
    "comparable_with_caveat",
    "context_only",
    "not_comparable",
]:
    print(f"- {status}: {status_counts.get(status, 0)}")

print(
    "\nKey Stage 2F result: "
    "no assessed comparison is directly comparable without qualification."
)
print(
    f"Wrote {summary_path.relative_to(REPO_DIR)}"
)


Stage 2F validation passed.
Overall status: PASS_WITH_CAVEAT
Comparability assessments: 17
Status counts:
- directly_comparable: 0
- comparable_with_transformation: 3
- comparable_with_caveat: 4
- context_only: 3
- not_comparable: 7

Key Stage 2F result: no assessed comparison is directly comparable without qualification.
Wrote metadata/stage2_comparability_summary.csv



# Stage 2G — Extraction Validation

Stage 2G validates the Stage 2 extraction and standardization outputs before Stage 2 closure.

The gate checks:

- file and row-count integrity;
- observation identity and source lineage;
- schema completeness;
- numeric and denominator consistency;
- semantic mapping safeguards;
- temporal evidence classification;
- evidence-layer separation;
- absence of premature analytical transformations;
- Stage 2F comparability integrity;
- duplicate-evidence controls;
- whether the available evidence supports a complete project unit-economics chain.

A validation caveat is not treated as a failure when it reflects a real limitation of the source evidence that has been preserved explicitly.



## 16. Reload Stage 2 outputs for validation

**Purpose:** load the standardized observations, output manifest, mapping registry, reference inputs, and comparability assessment from disk so Stage 2G validates persisted outputs rather than only in-memory objects.


In [16]:

stage2_standardized_files = sorted(STANDARDIZED_DIR.glob("src*_*.csv"))

persisted_rows = []
persisted_rows_by_file = {}

for path in stage2_standardized_files:
    rows = read_csv(path)
    persisted_rows_by_file[path.name] = rows
    persisted_rows.extend(rows)

persisted_manifest = read_csv(
    METADATA_DIR / "stage2_standardized_output_manifest.csv"
)
persisted_mapping = read_csv(
    METADATA_DIR / "stage2_metric_mapping.csv"
)
persisted_references = read_csv(
    METADATA_DIR / "stage2_reference_input_registry.csv"
)
persisted_comparability = read_csv(
    METADATA_DIR / "stage2_comparability_assessment.csv"
)
persisted_comparability_summary = read_csv(
    METADATA_DIR / "stage2_comparability_summary.csv"
)

persisted_observation_lookup = {
    row["observation_id"]: row
    for row in persisted_rows
}

print(f"Persisted standardized files: {len(stage2_standardized_files)}")
print(f"Persisted standardized observations: {len(persisted_rows)}")
print(f"Persisted comparability assessments: {len(persisted_comparability)}")
print(f"Persisted reference inputs: {len(persisted_references)}")


Persisted standardized files: 5
Persisted standardized observations: 72
Persisted comparability assessments: 17
Persisted reference inputs: 6



## 17. Run the Stage 2G extraction-validation gate

**Purpose:** execute a fixed set of structural, numerical, semantic, temporal, and comparability checks. `PASS` means the check is satisfied; `CAVEAT` means the output is valid but a source limitation remains binding; `FAIL` blocks Stage 2 closure.


In [17]:

VALIDATION_COLUMNS = [
    "check_id",
    "check_name",
    "status",
    "severity",
    "evidence",
    "analytical_implication",
]

validation_rows = []

def add_validation(check_id, check_name, status, severity, evidence, implication):
    validation_rows.append({
        "check_id": check_id,
        "check_name": check_name,
        "status": status,
        "severity": severity,
        "evidence": evidence,
        "analytical_implication": implication,
    })

def numeric_or_none(value):
    if value in ("", None):
        return None
    return float(value)

# S2V001 — expected standardized files exist.
expected_standardized_filenames = {
    "src010_polling_institute_2022_driver_observations.csv",
    "src013_ideas_2025_driver_observations.csv",
    "src017_indef_pppi_2025_driver_context_observations.csv",
    "src029_ideas_2023_driver_observations.csv",
    "src030_ideas_2020_driver_observations.csv",
}
actual_standardized_filenames = {
    path.name for path in stage2_standardized_files
}
missing_standardized_files = sorted(
    expected_standardized_filenames - actual_standardized_filenames
)
unexpected_standardized_files = sorted(
    actual_standardized_filenames - expected_standardized_filenames
)
add_validation(
    "S2V001",
    "Expected standardized source files are present",
    "PASS" if not missing_standardized_files and not unexpected_standardized_files else "FAIL",
    "blocking",
    (
        f"Expected={len(expected_standardized_filenames)}; "
        f"found={len(actual_standardized_filenames)}; "
        f"missing={missing_standardized_files}; unexpected={unexpected_standardized_files}"
    ),
    "Stage 2 closure requires the validated source-specific output set.",
)

# S2V002 — output manifest counts match persisted row counts.
manifest_count_errors = []
for row in persisted_manifest:
    output_path = row["output_path"].strip()
    if not output_path:
        continue
    filename = Path(output_path).name
    expected_count = int(row["observation_count"])
    actual_count = len(persisted_rows_by_file.get(filename, []))
    if expected_count != actual_count:
        manifest_count_errors.append(
            f"{row['source_id']} expected {expected_count}, found {actual_count}"
        )
add_validation(
    "S2V002",
    "Output manifest row counts match persisted files",
    "PASS" if not manifest_count_errors else "FAIL",
    "blocking",
    "No mismatches." if not manifest_count_errors else "; ".join(manifest_count_errors),
    "Manifest counts must be reproducible from persisted standardized files.",
)

# S2V003 — total observation count.
add_validation(
    "S2V003",
    "Validated Stage 2E observation count is preserved",
    "PASS" if len(persisted_rows) == 72 else "FAIL",
    "blocking",
    f"Persisted observations={len(persisted_rows)}; expected=72.",
    "The Stage 2E extraction baseline must remain stable through Stage 2F and Stage 2G.",
)

# S2V004 — unique observation IDs.
observation_ids = [row["observation_id"] for row in persisted_rows]
duplicate_observation_ids = sorted(
    observation_id
    for observation_id, count in Counter(observation_ids).items()
    if count > 1
)
add_validation(
    "S2V004",
    "Observation identifiers are unique",
    "PASS" if not duplicate_observation_ids else "FAIL",
    "blocking",
    f"Duplicate IDs={duplicate_observation_ids}",
    "Duplicate observation IDs would break source-to-analysis traceability.",
)

# S2V005 — all standardized source IDs are registered.
unregistered_sources = sorted({
    row["source_id"]
    for row in persisted_rows
    if row["source_id"] not in registered_source_ids
})
add_validation(
    "S2V005",
    "All standardized observations resolve to registered sources",
    "PASS" if not unregistered_sources else "FAIL",
    "blocking",
    f"Unregistered source IDs={unregistered_sources}",
    "Every standardized observation must trace to the canonical source registry.",
)

# S2V006 — exact standardized schema.
schema_field_rows = read_csv(
    METADATA_DIR / "stage2_extraction_schema.csv"
)
expected_schema_columns = [row["field_name"] for row in schema_field_rows]
schema_errors = []
for path in stage2_standardized_files:
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        fieldnames = csv.DictReader(f).fieldnames or []
    if fieldnames != expected_schema_columns:
        schema_errors.append(path.name)
add_validation(
    "S2V006",
    "Standardized files use the locked Stage 2 extraction schema",
    "PASS" if not schema_errors else "FAIL",
    "blocking",
    f"Files with schema mismatch={schema_errors}",
    "A common source-preserving schema is required before Stage 3 harmonization.",
)

# S2V007 — required fields populated.
required_schema_fields = {
    row["field_name"]
    for row in schema_field_rows
    if row["required"].strip().lower() == "yes"
}
required_field_exceptions = {
    # value_numeric is required in the current standardized numerical dataset.
}
missing_required_values = []
for row in persisted_rows:
    for field in required_schema_fields:
        if field in required_field_exceptions:
            continue
        if row.get(field, "") in ("", None):
            missing_required_values.append(
                f"{row['observation_id']}:{field}"
            )
add_validation(
    "S2V007",
    "Required standardized fields are populated",
    "PASS" if not missing_required_values else "FAIL",
    "blocking",
    (
        "No missing required fields."
        if not missing_required_values
        else f"Missing required values={missing_required_values[:20]}"
    ),
    "Required identity, semantic, evidence, and traceability fields cannot be blank.",
)

# S2V008 — numeric values parse.
non_numeric_values = []
for row in persisted_rows:
    try:
        float(row["value_numeric"])
    except (TypeError, ValueError):
        non_numeric_values.append(row["observation_id"])
add_validation(
    "S2V008",
    "All standardized quantitative values are numeric",
    "PASS" if not non_numeric_values else "FAIL",
    "blocking",
    f"Non-numeric value rows={non_numeric_values}",
    "Stage 3 cannot safely process malformed quantitative values.",
)

# S2V009 — percentage values bounded.
invalid_percentages = []
for row in persisted_rows:
    if row["unit"] == "percent":
        value = float(row["value_numeric"])
        if not 0 <= value <= 100:
            invalid_percentages.append(
                f"{row['observation_id']}={value}"
            )
add_validation(
    "S2V009",
    "Percentage observations fall within 0-100",
    "PASS" if not invalid_percentages else "FAIL",
    "blocking",
    f"Invalid percentages={invalid_percentages}",
    "Out-of-range percentages would indicate extraction or standardization error.",
)

# S2V010 — category bounds internally ordered.
invalid_bounds = []
for row in persisted_rows:
    lower = numeric_or_none(row["category_lower_bound"])
    upper = numeric_or_none(row["category_upper_bound"])
    if lower is not None and upper is not None and lower > upper:
        invalid_bounds.append(row["observation_id"])
add_validation(
    "S2V010",
    "Category bounds are internally ordered",
    "PASS" if not invalid_bounds else "FAIL",
    "blocking",
    f"Invalid lower/upper bounds={invalid_bounds}",
    "Distribution brackets must retain a valid numerical ordering.",
)

# S2V011 — metric denominators cannot exceed the source sample size.
invalid_denominators = []
unresolved_denominator_count = 0
for row in persisted_rows:
    denominator = numeric_or_none(row["metric_denominator_n"])
    sample_size = numeric_or_none(row["sample_size"])
    if denominator is None:
        unresolved_denominator_count += 1
    elif sample_size is not None and denominator > sample_size:
        invalid_denominators.append(
            f"{row['observation_id']} n={denominator} > sample={sample_size}"
        )
add_validation(
    "S2V011",
    "Reported metric denominators do not exceed source sample sizes",
    "PASS" if not invalid_denominators else "FAIL",
    "blocking",
    (
        f"No impossible denominators; unresolved metric-specific n="
        f"{unresolved_denominator_count}."
    )
    if not invalid_denominators
    else "; ".join(invalid_denominators),
    "Blank metric-specific denominators remain missing rather than being inferred.",
)

# S2V012 — project metrics resolve to locked Stage 0 metrics.
invalid_project_metrics = sorted({
    row["project_metric"]
    for row in persisted_rows
    if row["project_metric"]
    and row["project_metric"] not in project_metrics
})
add_validation(
    "S2V012",
    "Mapped project metrics resolve to the locked Stage 0 dictionary",
    "PASS" if not invalid_project_metrics else "FAIL",
    "blocking",
    f"Invalid project metrics={invalid_project_metrics}",
    "Stage 2 must not introduce undeclared project metrics through silent relabelling.",
)

# S2V013 — ambiguous/source-defined income remains unmapped.
ambiguous_income_codes = {
    "source_reported_daily_income_unspecified_layer",
    "source_defined_net_after_mixed_cost_bundle",
    "source_defined_net_income_unspecified_cost_basis",
}
incorrect_income_mapping = [
    row["observation_id"]
    for row in persisted_rows
    if row["source_metric_code"] in ambiguous_income_codes
    and row["project_metric"] != ""
]
add_validation(
    "S2V013",
    "Ambiguous and source-defined income concepts remain unmapped",
    "PASS" if not incorrect_income_mapping else "FAIL",
    "blocking",
    f"Incorrectly mapped rows={incorrect_income_mapping}",
    "Generic or source-defined net income cannot be relabelled as project gross, receipts, or net operating earnings.",
)

# S2V014 — mixed fuel-food cost boundary remains explicit and unmapped.
mixed_cost_codes = {
    "fuel_food_daily_cost_bundle",
    "fuel_food_cost_share_of_gross",
}
mixed_cost_errors = [
    row["observation_id"]
    for row in persisted_rows
    if row["source_metric_code"] in mixed_cost_codes
    and (
        row["cost_boundary"] != "mixed_work_personal"
        or row["project_metric"] != ""
    )
]
add_validation(
    "S2V014",
    "Mixed fuel-plus-food cost bundles stay outside project operating costs",
    "PASS" if not mixed_cost_errors else "FAIL",
    "blocking",
    f"Boundary/mapping violations={mixed_cost_errors}",
    "Personal food/drink expenditure must not enter project operating-cost totals.",
)

# S2V015 — deduction evidence is not converted to realized transaction deduction.
deduction_mapping_errors = [
    row["observation_id"]
    for row in persisted_rows
    if row["source_metric_code"] == "driver_reported_deduction_rate_category"
    and row["project_metric"] == "realized_driver_deduction_rate"
]
add_validation(
    "S2V015",
    "Driver-reported deduction categories remain separate from realized deductions",
    "PASS" if not deduction_mapping_errors else "FAIL",
    "blocking",
    f"Incorrect realized-deduction mappings={deduction_mapping_errors}",
    "Survey-reported deduction categories cannot substitute for matched transaction evidence.",
)

# S2V016 — generic working time maps only to source-reported working hours.
working_time_errors = [
    row["observation_id"]
    for row in persisted_rows
    if row["source_metric_code"] == "generic_working_hours"
    and (
        row["project_metric"] != "working_hours_source_reported"
        or row["working_time_basis"] != "source_reported_unspecified"
    )
]
add_validation(
    "S2V016",
    "Generic working time is not relabelled as online or productive time",
    "PASS" if not working_time_errors else "FAIL",
    "blocking",
    f"Working-time mapping violations={working_time_errors}",
    "Hourly analysis must preserve the source-reported time-basis caveat.",
)

# S2V017 — ambiguous distance remains unmapped.
distance_mapping_errors = [
    row["observation_id"]
    for row in persisted_rows
    if row["source_metric_code"] == "daily_distance_source_reported"
    and (
        row["project_metric"] != ""
        or row["distance_basis"] != "source_reported_unspecified"
    )
]
add_validation(
    "S2V017",
    "Ambiguous daily distance is not treated as total work-related distance",
    "PASS" if not distance_mapping_errors else "FAIL",
    "blocking",
    f"Distance mapping violations={distance_mapping_errors}",
    "Fuel and per-km economics cannot silently use ambiguous distance as total work distance.",
)

# S2V018 — recalled historical values retain temporal status.
recall_errors = [
    row["observation_id"]
    for row in persisted_rows
    if (
        row["source_metric_code"] == "explicit_gross_driver_earnings_recalled"
        and row["temporal_evidence_status"] != "retrospective_recall"
    )
    or (
        row["source_id"] == "SRC010"
        and row["observation_id"] in {
            f"SRC010-O{i:03d}" for i in range(1, 13)
        }
        and row["temporal_evidence_status"] != "retrospective_recall"
    )
]
add_validation(
    "S2V018",
    "Retrospective historical values remain classified as recall",
    "PASS" if not recall_errors else "FAIL",
    "blocking",
    f"Temporal-classification violations={recall_errors}",
    "Recall values must not be interpreted as independent historical survey waves.",
)

# S2V019 — Stage 2E rows remain untransformed.
unexpected_transformations = [
    row["observation_id"]
    for row in persisted_rows
    if row["transformation_applied"] != "none"
    or row["transformation_formula"] not in ("", None)
]
add_validation(
    "S2V019",
    "Stage 2 standardized observations contain no analytical transformation",
    "PASS" if not unexpected_transformations else "FAIL",
    "blocking",
    f"Unexpected transformed rows={unexpected_transformations}",
    "Inflation, modelling, and analytical transformations belong to later documented stages.",
)

# S2V020 — no global observation-level comparability tags.
global_comparability_errors = [
    row["observation_id"]
    for row in persisted_rows
    if row["comparability_use"] not in ("", None)
    or row["comparability_status"] not in ("", None)
]
add_validation(
    "S2V020",
    "Comparability remains use-specific rather than globally attached to observations",
    "PASS" if not global_comparability_errors else "FAIL",
    "blocking",
    f"Premature/global comparability tags={global_comparability_errors}",
    "The same observation may have different eligibility across analytical uses.",
)

# S2V021 — comparability statuses and observation references valid.
allowed_statuses = {
    "directly_comparable",
    "comparable_with_transformation",
    "comparable_with_caveat",
    "context_only",
    "not_comparable",
}
comparability_errors = []
for row in persisted_comparability:
    if row["comparability_status"] not in allowed_statuses:
        comparability_errors.append(
            f"{row['comparison_id']}:invalid_status"
        )
    for observation_id in row["observation_ids"].split(";"):
        if observation_id not in persisted_observation_lookup:
            comparability_errors.append(
                f"{row['comparison_id']}:missing:{observation_id}"
            )
add_validation(
    "S2V021",
    "Stage 2F comparability assessments use valid statuses and traceable observations",
    "PASS" if not comparability_errors else "FAIL",
    "blocking",
    f"Comparability integrity errors={comparability_errors}",
    "Stage 3 must be able to trace every comparison decision to standardized observations.",
)

# S2V022 — transformations required where status says so.
missing_required_transformations = [
    row["comparison_id"]
    for row in persisted_comparability
    if row["comparability_status"] == "comparable_with_transformation"
    and row["required_transformation"].strip().lower()
    in {"", "none", "not applicable."}
]
add_validation(
    "S2V022",
    "Comparable-with-transformation assessments specify the required transformation",
    "PASS" if not missing_required_transformations else "FAIL",
    "blocking",
    f"Missing transformation instructions={missing_required_transformations}",
    "Later analysis cannot use transformed-comparable evidence without a documented transformation.",
)

# S2V023 — comparability summary reproduces assessment counts.
actual_comparability_counts = Counter(
    row["comparability_status"] for row in persisted_comparability
)
summary_row = persisted_comparability_summary[0]
comparability_summary_errors = []
summary_count_map = {
    "directly_comparable": "directly_comparable_count",
    "comparable_with_transformation": "comparable_with_transformation_count",
    "comparable_with_caveat": "comparable_with_caveat_count",
    "context_only": "context_only_count",
    "not_comparable": "not_comparable_count",
}
for status, summary_field in summary_count_map.items():
    if int(summary_row[summary_field]) != actual_comparability_counts.get(status, 0):
        comparability_summary_errors.append(status)
if int(summary_row["assessment_count"]) != len(persisted_comparability):
    comparability_summary_errors.append("assessment_count")
add_validation(
    "S2V023",
    "Stage 2F summary reproduces the persisted comparability assessment",
    "PASS" if not comparability_summary_errors else "FAIL",
    "blocking",
    (
        f"Assessment count={len(persisted_comparability)}; "
        f"status counts={dict(actual_comparability_counts)}; "
        f"mismatches={comparability_summary_errors}"
    ),
    "Summary counts must be mechanically reproducible from the assessment table.",
)

# S2V024 — SRC010 full distributions sum to 100 within one-decimal rounding tolerance.
src010_rows_sorted = sorted(
    [row for row in persisted_rows if row["source_id"] == "SRC010"],
    key=lambda row: row["observation_id"],
)
src010_period_groups = [
    src010_rows_sorted[0:6],
    src010_rows_sorted[6:12],
    src010_rows_sorted[12:18],
]
src010_sums = [
    sum(float(row["value_numeric"]) for row in group)
    for group in src010_period_groups
]
src010_distribution_errors = [
    total for total in src010_sums
    if abs(total - 100.0) > 0.2
]
add_validation(
    "S2V024",
    "Complete SRC010 income distributions reconcile to 100 percent within rounding tolerance",
    "PASS" if not src010_distribution_errors else "FAIL",
    "blocking",
    f"Distribution sums={[round(total, 3) for total in src010_sums]}; tolerance=±0.2 percentage points.",
    "Small 99.9/100.1 totals are accepted as one-decimal source rounding, not corrected.",
)

# S2V025 — unresolved metric-specific denominators are preserved as missing.
add_validation(
    "S2V025",
    "Unresolved metric-specific denominators remain explicit",
    "CAVEAT" if unresolved_denominator_count > 0 else "PASS",
    "non_blocking",
    f"Observations with unresolved metric-specific denominator n={unresolved_denominator_count}.",
    "Analyses requiring respondent counts must exclude or caveat these rows rather than assume the full sample denominator.",
)

# S2V026 — IDEAS 62/67 geography discrepancy remains explicit.
src013_geographies = {
    row["geography"]
    for row in persisted_rows
    if row["source_id"] == "SRC013"
}
geography_discrepancy_preserved = any(
    "62" in geography and "67" in geography
    for geography in src013_geographies
)
add_validation(
    "S2V026",
    "SRC013 internal 62-versus-67 locality discrepancy is preserved",
    "CAVEAT" if geography_discrepancy_preserved else "FAIL",
    "non_blocking" if geography_discrepancy_preserved else "blocking",
    f"SRC013 geography values={sorted(src013_geographies)}",
    "No single locality count may be asserted as certain until the source discrepancy is independently resolved.",
)

# S2V027 — overlapping INDEF companion source is not duplicated.
src025_rows = [
    row for row in persisted_rows
    if row["source_id"] == "SRC025"
]
manifest_src025 = [
    row for row in persisted_manifest
    if row["source_id"] == "SRC025"
]
src025_control_ok = (
    not src025_rows
    and len(manifest_src025) == 1
    and int(manifest_src025[0]["observation_count"]) == 0
)
add_validation(
    "S2V027",
    "SRC025 companion presentation is not double-counted against SRC017",
    "PASS" if src025_control_ok else "FAIL",
    "blocking",
    (
        f"SRC025 standardized rows={len(src025_rows)}; "
        f"manifest entries={len(manifest_src025)}."
    ),
    "Overlapping figures from the same INDEF-PPPI programme must not become independent observations.",
)

# S2V028 — no project net operating earnings are manufactured in Stage 2.
project_net_rows = [
    row["observation_id"]
    for row in persisted_rows
    if row["project_metric"] in {
        "net_operating_earnings_cash_basis",
        "net_operating_earnings_economic_basis",
        "net_earnings_per_online_hour",
        "net_earnings_per_productive_hour",
        "net_earnings_per_order",
        "net_earnings_per_km",
    }
]
add_validation(
    "S2V028",
    "Stage 2 does not manufacture project net operating earnings",
    "PASS" if not project_net_rows else "FAIL",
    "blocking",
    f"Project net-derived rows={project_net_rows}",
    "Net operating earnings require a defensible receipts-and-cost chain that Stage 2 does not yet contain.",
)

# S2V029 — reference inputs resolve to registered source IDs and remain separate.
invalid_reference_sources = sorted({
    row["source_id"]
    for row in persisted_references
    if row["source_id"] not in registered_source_ids
})
add_validation(
    "S2V029",
    "Stage 2 reference inputs retain valid provenance and separate evidence roles",
    "PASS" if not invalid_reference_sources else "FAIL",
    "blocking",
    f"Reference records={len(persisted_references)}; invalid source IDs={invalid_reference_sources}",
    "CPI, fuel-price, regulatory, and platform-policy inputs remain reference evidence rather than driver observations.",
)

# S2V030 — no assessed comparison is directly comparable.
direct_comparisons = [
    row["comparison_id"]
    for row in persisted_comparability
    if row["comparability_status"] == "directly_comparable"
]
add_validation(
    "S2V030",
    "Cross-source comparability limitations remain binding",
    "CAVEAT" if not direct_comparisons else "PASS",
    "non_blocking",
    (
        "No Stage 2F comparison is directly comparable without transformation, caveat, contextual restriction, or exclusion."
        if not direct_comparisons
        else f"Directly comparable assessments={direct_comparisons}"
    ),
    "Stage 3 and later analysis must honor the use-specific comparability assessment rather than pool all standardized evidence.",
)

# S2V031 — same-observation unit-economics chain remains incomplete.
available_project_metrics = {
    row["project_metric"]
    for row in persisted_rows
    if row["project_metric"]
}
chain_required = {
    "driver_gross_service_earnings",
    "driver_side_platform_deduction",
    "driver_receipts_before_operating_cost",
    "fuel_cost",
}
missing_chain_metrics = sorted(chain_required - available_project_metrics)
add_validation(
    "S2V031",
    "Complete same-observation project unit-economics chain is not falsely asserted",
    "CAVEAT" if missing_chain_metrics else "PASS",
    "non_blocking",
    f"Missing standardized project-chain metrics={missing_chain_metrics}",
    "Stage 4 may analyze available components and scenarios, but must not present an observed gross-to-net chain that the evidence does not contain.",
)

# S2V032 — no blocking failures.
blocking_failures = [
    row["check_id"]
    for row in validation_rows
    if row["status"] == "FAIL"
]
add_validation(
    "S2V032",
    "No blocking extraction-validation failure remains",
    "PASS" if not blocking_failures else "FAIL",
    "blocking",
    f"Blocking failures before final gate={blocking_failures}",
    "Stage 2H may proceed only when Stage 2G has no blocking failure.",
)

validation_path = METADATA_DIR / "stage2_extraction_validation.csv"
write_csv(
    validation_path,
    validation_rows,
    VALIDATION_COLUMNS,
)

status_counts = Counter(row["status"] for row in validation_rows)

overall_status = (
    "FAIL"
    if status_counts.get("FAIL", 0) > 0
    else "PASS_WITH_CAVEAT"
    if status_counts.get("CAVEAT", 0) > 0
    else "PASS"
)

summary_rows = [{
    "stage": "Stage 2G",
    "validation_status": overall_status,
    "validation_check_count": len(validation_rows),
    "pass_count": status_counts.get("PASS", 0),
    "caveat_count": status_counts.get("CAVEAT", 0),
    "fail_count": status_counts.get("FAIL", 0),
    "standardized_observation_count": len(persisted_rows),
    "comparability_assessment_count": len(persisted_comparability),
    "reference_input_count": len(persisted_references),
    "closure_eligibility": (
        "eligible_for_stage2h"
        if overall_status in {"PASS", "PASS_WITH_CAVEAT"}
        else "blocked"
    ),
    "key_conclusion": (
        "Extraction and standardization integrity checks pass. "
        "Source-definition, denominator, geography, comparability, and "
        "unit-economics coverage caveats remain binding."
    ),
}]

validation_summary_path = (
    METADATA_DIR / "stage2_extraction_validation_summary.csv"
)
write_csv(
    validation_summary_path,
    summary_rows,
    [
        "stage",
        "validation_status",
        "validation_check_count",
        "pass_count",
        "caveat_count",
        "fail_count",
        "standardized_observation_count",
        "comparability_assessment_count",
        "reference_input_count",
        "closure_eligibility",
        "key_conclusion",
    ],
)

print("Stage 2G extraction-validation gate completed.")
print(f"Overall status: {overall_status}")
print(f"Checks: {len(validation_rows)}")
print(f"PASS: {status_counts.get('PASS', 0)}")
print(f"CAVEAT: {status_counts.get('CAVEAT', 0)}")
print(f"FAIL: {status_counts.get('FAIL', 0)}")
print(f"Closure eligibility: {summary_rows[0]['closure_eligibility']}")
print(f"Wrote {validation_path.relative_to(REPO_DIR)}")
print(f"Wrote {validation_summary_path.relative_to(REPO_DIR)}")

if overall_status == "FAIL":
    failures = [
        f"{row['check_id']} — {row['check_name']}: {row['evidence']}"
        for row in validation_rows
        if row["status"] == "FAIL"
    ]
    raise RuntimeError(
        "Stage 2G validation failed:\n" + "\n".join(failures)
    )


Stage 2G extraction-validation gate completed.
Overall status: PASS_WITH_CAVEAT
Checks: 32
PASS: 28
CAVEAT: 4
FAIL: 0
Closure eligibility: eligible_for_stage2h
Wrote metadata/stage2_extraction_validation.csv
Wrote metadata/stage2_extraction_validation_summary.csv



## 18. Stage 2G validation result

**Purpose:** surface the persisted validation result and the non-blocking caveats that must remain binding in Stage 2H and subsequent analysis.


In [18]:

persisted_validation = read_csv(
    METADATA_DIR / "stage2_extraction_validation.csv"
)
persisted_validation_summary = read_csv(
    METADATA_DIR / "stage2_extraction_validation_summary.csv"
)[0]

caveat_rows = [
    row for row in persisted_validation
    if row["status"] == "CAVEAT"
]

print(
    f"Stage 2G status: "
    f"{persisted_validation_summary['validation_status']}"
)
print(
    f"Validation checks: "
    f"{persisted_validation_summary['validation_check_count']}"
)
print(
    f"Closure eligibility: "
    f"{persisted_validation_summary['closure_eligibility']}"
)

print("\nBinding caveats:")
for row in caveat_rows:
    print(
        f"- {row['check_id']} — {row['check_name']}: "
        f"{row['analytical_implication']}"
    )


Stage 2G status: PASS_WITH_CAVEAT
Validation checks: 32
Closure eligibility: eligible_for_stage2h

Binding caveats:
- S2V025 — Unresolved metric-specific denominators remain explicit: Analyses requiring respondent counts must exclude or caveat these rows rather than assume the full sample denominator.
- S2V026 — SRC013 internal 62-versus-67 locality discrepancy is preserved: No single locality count may be asserted as certain until the source discrepancy is independently resolved.
- S2V030 — Cross-source comparability limitations remain binding: Stage 3 and later analysis must honor the use-specific comparability assessment rather than pool all standardized evidence.
- S2V031 — Complete same-observation project unit-economics chain is not falsely asserted: Stage 4 may analyze available components and scenarios, but must not present an observed gross-to-net chain that the evidence does not contain.



# Stage 2H — Stage 2 Closure

Stage 2H closes the extraction and standardization stage by confirming that:

- Stage 2E standardized outputs are present and stable;
- Stage 2F comparability assessments are complete for the intended Stage 3–5 uses identified so far;
- Stage 2G has no blocking validation failure;
- remaining caveats are preserved as methodological constraints rather than silently resolved;
- no project net operating earnings, realized deduction rate, fuel model, inflation-adjusted value, or pooled analytical dataset has been manufactured prematurely.

The closure decision is a stage gate. It does not convert caveated evidence into stronger evidence.



## 19. Verify Stage 2 closure prerequisites

**Purpose:** reload persisted Stage 2 outputs and verify the minimum artifact set and gate conditions required to close Stage 2.


In [19]:

required_stage2_artifacts = [
    METADATA_DIR / "stage2_extraction_schema.csv",
    METADATA_DIR / "stage2_metric_mapping.csv",
    METADATA_DIR / "stage2_reference_input_registry.csv",
    METADATA_DIR / "stage2_methodological_decision_log.csv",
    METADATA_DIR / "stage2_standardized_output_manifest.csv",
    METADATA_DIR / "stage2_comparability_assessment.csv",
    METADATA_DIR / "stage2_comparability_summary.csv",
    METADATA_DIR / "stage2_extraction_validation.csv",
    METADATA_DIR / "stage2_extraction_validation_summary.csv",
]

missing_stage2_artifacts = [
    str(path.relative_to(REPO_DIR))
    for path in required_stage2_artifacts
    if not path.is_file()
]

stage2g_summary = read_csv(
    METADATA_DIR / "stage2_extraction_validation_summary.csv"
)[0]

stage2f_summary = read_csv(
    METADATA_DIR / "stage2_comparability_summary.csv"
)[0]

stage2_manifest = read_csv(
    METADATA_DIR / "stage2_standardized_output_manifest.csv"
)

stage2_validation = read_csv(
    METADATA_DIR / "stage2_extraction_validation.csv"
)

stage2_comparability = read_csv(
    METADATA_DIR / "stage2_comparability_assessment.csv"
)

standardized_stage2_files = sorted(
    STANDARDIZED_DIR.glob("src*_*.csv")
)

standardized_stage2_rows = []
for path in standardized_stage2_files:
    standardized_stage2_rows.extend(read_csv(path))

blocking_validation_failures = [
    row["check_id"]
    for row in stage2_validation
    if row["status"] == "FAIL"
]

if missing_stage2_artifacts:
    raise RuntimeError(
        "Stage 2 closure is blocked because required artifacts are missing: "
        + ", ".join(missing_stage2_artifacts)
    )

if stage2g_summary["validation_status"] not in {"PASS", "PASS_WITH_CAVEAT"}:
    raise RuntimeError(
        "Stage 2 closure is blocked because Stage 2G is not eligible for closure."
    )

if blocking_validation_failures:
    raise RuntimeError(
        "Stage 2 closure is blocked by validation failures: "
        + ", ".join(blocking_validation_failures)
    )

print("Stage 2 closure prerequisites passed.")
print(f"Standardized observations: {len(standardized_stage2_rows)}")
print(f"Comparability assessments: {len(stage2_comparability)}")
print(
    f"Stage 2G validation status: "
    f"{stage2g_summary['validation_status']}"
)
print(
    f"Stage 2F comparability status: "
    f"{stage2f_summary['stage2f_status']}"
)


Stage 2 closure prerequisites passed.
Standardized observations: 72
Comparability assessments: 17
Stage 2G validation status: PASS_WITH_CAVEAT
Stage 2F comparability status: PASS_WITH_CAVEAT



## 20. Record Stage 2 closure decisions

**Purpose:** preserve the final methodological implications of Stage 2 for later cleaning, harmonization, analysis, and reporting.


In [20]:

stage2h_decisions = [
    {
        "decision_id": "S2D019",
        "decision": "Close Stage 2 with source-specific standardized files rather than a pooled analytical dataset.",
        "rationale": "Stage 2F confirms that no assessed cross-source comparison is directly comparable without transformation, caveat, contextual restriction, or exclusion.",
        "evidence_basis": "Stage 2F comparability assessment; Stage 2G extraction validation",
        "analytical_implication": "Stage 3 must perform cleaning, harmonization, transformation eligibility checks, and analytical dataset preparation before any cross-source pooling.",
        "intended_report_destination": "Methodology — data flow and comparability",
    },
    {
        "decision_id": "S2D020",
        "decision": "Carry Stage 2 evidence limitations forward as binding analytical constraints.",
        "rationale": "Unresolved denominators, the SRC013 locality discrepancy, non-equivalent samples, retrospective recall, ambiguous time/distance concepts, and incomplete unit-economics chains are properties of the evidence rather than extraction defects.",
        "evidence_basis": "Stage 2G validation caveats",
        "analytical_implication": "Later stages may transform or restrict eligible evidence but must not silently eliminate these limitations.",
        "intended_report_destination": "Methodology — limitations and analytical eligibility",
    },
    {
        "decision_id": "S2D021",
        "decision": "Do not calculate project net operating earnings during Stage 2.",
        "rationale": "The available standardized evidence does not provide a complete same-observation chain from gross service earnings through realized driver deductions, receipts, and complete project-defined operating costs.",
        "evidence_basis": "S2V028; S2V031; Stage 0 unit-economics accounting framework",
        "analytical_implication": "Observed net operating earnings cannot be claimed from Stage 2 evidence; later scenario or reconstructed estimates require explicit documented assumptions and eligibility checks.",
        "intended_report_destination": "Methodology — unit-economics construction",
    },
]

existing_decisions = read_csv(
    METADATA_DIR / "stage2_methodological_decision_log.csv"
)

stage2h_ids = {
    row["decision_id"] for row in stage2h_decisions
}

existing_decisions = [
    row for row in existing_decisions
    if row["decision_id"] not in stage2h_ids
]

updated_decisions = existing_decisions + stage2h_decisions

write_csv(
    METADATA_DIR / "stage2_methodological_decision_log.csv",
    updated_decisions,
    [
        "decision_id",
        "decision",
        "rationale",
        "evidence_basis",
        "analytical_implication",
        "intended_report_destination",
    ],
)

print(
    "Stage 2 methodological decision log updated: "
    f"{len(updated_decisions)} total decisions."
)


Stage 2 methodological decision log updated: 21 total decisions.



## 21. Build the Stage 2 closure summary

**Purpose:** produce the formal Stage 2 gate result with quantitative output counts, comparability status distribution, validation status distribution, binding caveats, and the next-stage decision.


In [21]:

validation_status_counts = Counter(
    row["status"] for row in stage2_validation
)

comparability_status_counts = Counter(
    row["comparability_status"] for row in stage2_comparability
)

source_observation_counts = Counter(
    row["source_id"] for row in standardized_stage2_rows
)

binding_caveats = [
    row["analytical_implication"]
    for row in stage2_validation
    if row["status"] == "CAVEAT"
]

expected_source_counts = {
    "SRC010": 18,
    "SRC013": 18,
    "SRC017": 16,
    "SRC029": 14,
    "SRC030": 6,
}

if dict(source_observation_counts) != expected_source_counts:
    raise RuntimeError(
        "Stage 2 closure count mismatch. "
        f"Expected {expected_source_counts}; found {dict(source_observation_counts)}."
    )

if len(standardized_stage2_rows) != 72:
    raise RuntimeError(
        f"Stage 2 closure expected 72 standardized observations, found {len(standardized_stage2_rows)}."
    )

if len(stage2_comparability) != 17:
    raise RuntimeError(
        f"Stage 2 closure expected 17 comparability assessments, found {len(stage2_comparability)}."
    )

if validation_status_counts.get("FAIL", 0) != 0:
    raise RuntimeError(
        "Stage 2 closure is blocked because extraction validation contains FAIL."
    )

closure_status = (
    "PASS_WITH_CAVEAT"
    if validation_status_counts.get("CAVEAT", 0) > 0
    or comparability_status_counts.get("directly_comparable", 0) == 0
    else "PASS"
)

closure_rows = [{
    "stage": "Stage 2",
    "stage_title": "Data Extraction and Standardization",
    "closure_status": closure_status,
    "standardized_source_file_count": len(standardized_stage2_files),
    "standardized_observation_count": len(standardized_stage2_rows),
    "reference_input_count": len(read_csv(METADATA_DIR / "stage2_reference_input_registry.csv")),
    "comparability_assessment_count": len(stage2_comparability),
    "directly_comparable_count": comparability_status_counts.get("directly_comparable", 0),
    "comparable_with_transformation_count": comparability_status_counts.get("comparable_with_transformation", 0),
    "comparable_with_caveat_count": comparability_status_counts.get("comparable_with_caveat", 0),
    "context_only_count": comparability_status_counts.get("context_only", 0),
    "not_comparable_count": comparability_status_counts.get("not_comparable", 0),
    "validation_check_count": len(stage2_validation),
    "validation_pass_count": validation_status_counts.get("PASS", 0),
    "validation_caveat_count": validation_status_counts.get("CAVEAT", 0),
    "validation_fail_count": validation_status_counts.get("FAIL", 0),
    "next_stage": "Stage 3 — Data Cleaning, Harmonization and Validation",
    "next_stage_eligibility": "eligible",
    "key_conclusion": (
        "Stage 2 extraction, semantic standardization, comparability tagging, "
        "and validation are complete. The evidence is suitable for Stage 3 "
        "with the documented source-definition, sampling, denominator, "
        "temporal, geography, comparability, and unit-economics caveats retained."
    ),
    "binding_caveats": " | ".join(binding_caveats),
}]

closure_path = METADATA_DIR / "stage2_closure_summary.csv"
write_csv(
    closure_path,
    closure_rows,
    [
        "stage",
        "stage_title",
        "closure_status",
        "standardized_source_file_count",
        "standardized_observation_count",
        "reference_input_count",
        "comparability_assessment_count",
        "directly_comparable_count",
        "comparable_with_transformation_count",
        "comparable_with_caveat_count",
        "context_only_count",
        "not_comparable_count",
        "validation_check_count",
        "validation_pass_count",
        "validation_caveat_count",
        "validation_fail_count",
        "next_stage",
        "next_stage_eligibility",
        "key_conclusion",
        "binding_caveats",
    ],
)

print("Stage 2 closure summary written.")
print(f"Closure status: {closure_status}")
print(f"Standardized observations: {len(standardized_stage2_rows)}")
print(f"Comparability assessments: {len(stage2_comparability)}")
print(f"Validation checks: {len(stage2_validation)}")
print("Next-stage eligibility: eligible")
print(f"Wrote {closure_path.relative_to(REPO_DIR)}")


Stage 2 closure summary written.
Closure status: PASS_WITH_CAVEAT
Standardized observations: 72
Comparability assessments: 17
Validation checks: 32
Next-stage eligibility: eligible
Wrote metadata/stage2_closure_summary.csv



## 22. Validate Stage 2 closure

**Purpose:** verify that the final closure summary is mechanically consistent with persisted Stage 2 outputs and that no blocking condition remains before transition to Stage 3.


In [22]:

closure_summary = read_csv(
    METADATA_DIR / "stage2_closure_summary.csv"
)[0]

closure_checks = []

def closure_check(check_id, name, condition, evidence):
    closure_checks.append({
        "check_id": check_id,
        "check_name": name,
        "status": "PASS" if condition else "FAIL",
        "evidence": evidence,
    })

closure_check(
    "S2H001",
    "Stage 2G permits closure",
    stage2g_summary["closure_eligibility"] == "eligible_for_stage2h",
    f"Stage 2G closure eligibility={stage2g_summary['closure_eligibility']}",
)

closure_check(
    "S2H002",
    "Stage 2 contains the validated 72 standardized observations",
    int(closure_summary["standardized_observation_count"]) == 72,
    f"Observed={closure_summary['standardized_observation_count']}",
)

closure_check(
    "S2H003",
    "Stage 2 contains 17 comparability assessments",
    int(closure_summary["comparability_assessment_count"]) == 17,
    f"Observed={closure_summary['comparability_assessment_count']}",
)

closure_check(
    "S2H004",
    "Stage 2 extraction validation has zero failures",
    int(closure_summary["validation_fail_count"]) == 0,
    f"FAIL count={closure_summary['validation_fail_count']}",
)

closure_check(
    "S2H005",
    "Stage 2 closure preserves the expected caveated status",
    closure_summary["closure_status"] == "PASS_WITH_CAVEAT",
    f"Closure status={closure_summary['closure_status']}",
)

closure_check(
    "S2H006",
    "Stage 3 transition is explicitly eligible",
    closure_summary["next_stage_eligibility"] == "eligible",
    f"Next-stage eligibility={closure_summary['next_stage_eligibility']}",
)

closure_failures = [
    row for row in closure_checks
    if row["status"] == "FAIL"
]

closure_validation_path = (
    METADATA_DIR / "stage2_closure_validation.csv"
)

write_csv(
    closure_validation_path,
    closure_checks,
    ["check_id", "check_name", "status", "evidence"],
)

print("Stage 2H closure validation completed.")
print(f"Checks: {len(closure_checks)}")
print(f"PASS: {len(closure_checks) - len(closure_failures)}")
print(f"FAIL: {len(closure_failures)}")
print(f"Wrote {closure_validation_path.relative_to(REPO_DIR)}")

if closure_failures:
    raise RuntimeError(
        "Stage 2H closure failed: "
        + "; ".join(
            f"{row['check_id']} {row['evidence']}"
            for row in closure_failures
        )
    )


Stage 2H closure validation completed.
Checks: 6
PASS: 6
FAIL: 0
Wrote metadata/stage2_closure_validation.csv



## 23. Stage 2 final gate

**Purpose:** display the final Stage 2 gate result used to authorize transition into Stage 3.


In [23]:

final_closure = read_csv(
    METADATA_DIR / "stage2_closure_summary.csv"
)[0]

print("=" * 72)
print("STAGE 2 — DATA EXTRACTION AND STANDARDIZATION")
print("=" * 72)
print(f"Closure status: {final_closure['closure_status']}")
print(
    f"Standardized observations: "
    f"{final_closure['standardized_observation_count']}"
)
print(
    f"Comparability assessments: "
    f"{final_closure['comparability_assessment_count']}"
)
print(
    f"Validation checks: "
    f"{final_closure['validation_check_count']}"
)
print(
    f"Validation failures: "
    f"{final_closure['validation_fail_count']}"
)
print(
    f"Next stage: "
    f"{final_closure['next_stage']}"
)
print(
    f"Next-stage eligibility: "
    f"{final_closure['next_stage_eligibility']}"
)
print("=" * 72)


STAGE 2 — DATA EXTRACTION AND STANDARDIZATION
Closure status: PASS_WITH_CAVEAT
Standardized observations: 72
Comparability assessments: 17
Validation checks: 32
Validation failures: 0
Next stage: Stage 3 — Data Cleaning, Harmonization and Validation
Next-stage eligibility: eligible
